# persian fairytale. V2 — запасная мастерская Python / PyTorch

Те же объяснения, упражнения и настоящие проверки, что в игре. Можно открыть в локальном Jupyter или Google Colab. Внешняя среда может потребовать установку PyTorch. Данные синтетические; прогресс notebook не переносится в игру автоматически. Сначала разберите объяснение, затем измените starter и запустите проверку. Эталон раскрывается отдельно.

In [ ]:
# При отсутствии PyTorch раскомментируйте: установка требует интернета.
# %pip install torch numpy
import sys, math, json
import torch
torch.set_num_threads(1)
torch.manual_seed(7)
print('Python', sys.version.split()[0], 'PyTorch', torch.__version__, 'CPU')


In [ ]:
# Самостоятельные проверочные случаи и построение графиков из настоящих вычислений.
EXERCISES = {'metrics': {'requiresTorch': False, 'hint': 'Recall отвечает: какую долю опасных пропусков обнаружили? Его знаменатель TP+FN. Precision: какая доля задержанных действительно опасна? TP+FP. FP — задержали честного, FN — пропустили опасного. F1 — гармоническое среднее, не обычное арифметическое.'}, 'regularization': {'requiresTorch': False, 'hint': 'Увеличение lambda уменьшает |w|. Это воздействие на обучение, а не метрика и не обещание меньшей ошибки. Слишком большая lambda ведёт к underfitting. Здесь SUM loss: при MEAN loss масштаб lambda отличался бы.'}, 'array_scan': {'requiresTorch': False, 'hint': 'Сначала проверьте: можно ли выгодно продать сегодня? Затем обновите минимальную цену. Ответ для [5, 4, 3] равен 0, а не -1.'}, 'tensors': {'requiresTorch': True, 'hint': 'Tensor — многомерный массив с операциями и возможностью autograd. Числа по умолчанию могут дать int64; для обучения здесь явно выберите torch.float32.'}, 'shapes': {'requiresTorch': True, 'hint': 'В shape [2, 3] первая ось перебирает два объекта, вторая — три признака. mean(dim=1) убирает ось признаков: остаётся shape [2]. Не превращайте tensor в список или numpy: так потеряется autograd.'}, 'autograd': {'requiresTorch': True, 'hint': 'Derivative x³−x² равна 3x²−2x. backward() идёт по computational graph и накапливает gradient в leaf tensor. В этом упражнении x свежий; при обучении старые gradients необходимо очищать.'}, 'module': {'requiresTorch': True, 'hint': 'nn.Module регистрирует вложенные слои и параметры. nn.Linear(2,1) хранит matrix weight [1,2] и bias [1]. Используйте model(x), а не прямой вызов forward: так сохраняются механизмы Module.'}, 'losses': {'requiresTorch': True, 'hint': 'Cross-entropy внутри использует log-softmax; повторный softmax изменит задачу. MSE сравнивает численные значения. Contrastive loss учит близости/различию представлений; он нужен для другой постановки, а не вместо любой loss.'}, 'optimizer': {'requiresTorch': True, 'hint': 'Порядок: zero_grad → forward/loss → backward → step. backward вычисляет gradients, step изменяет параметры. zero_grad очищает накопление с прошлого шага; model.train() его не заменяет.'}, 'modes': {'requiresTorch': True, 'hint': 'model.eval() не запрещает вычисление gradients. torch.no_grad() не выключает Dropout. Для этой проверки нужны оба: правильное поведение слоёв и отсутствие графа.'}, 'training': {'requiresTorch': True, 'hint': 'Вычислите loss внутри цикла по x_train,y_train. После цикла отдельно измерьте validation под no_grad. Успех на этой простой прямой не гарантирует generalization в другой задаче; эксперимент проверяет процедуру.'}, 'normalization': {'requiresTorch': True, 'hint': 'Sequential вызывает слои по порядку; число выходов одного Linear равно числу входов следующего. Dropout(p) обнуляет активации случайно только в train. evaluate_mse использует и eval(), и no_grad(): первое меняет поведение слоёв, второе отключает граф. LayerNorm нормирует строку, а не выключает элементы; это другой механизм.'}, 'two_sum': {'requiresTorch': False, 'hint': 'Число может повторяться: [3,3], target=6 → [0,1]. Нельзя использовать индекс дважды. Сначала ищите дополнение среди прошлых чисел, потом запоминайте текущее.'}, 'agent_loop': {'requiresTorch': False, 'hint': 'Условия остановки проверяются до следующего вызова. Даже если инструмент не сдвигает позицию, budget обязан завершить цикл. Результат инструмента — факт о мире; не подменяйте его ожиданием агента.'}, 'brackets': {'requiresTorch': False, 'hint': 'Stack хранит ожидаемые закрывающие скобки. Одних равных количеств недостаточно: ([)] имеет правильные количества, но неверную вложенность.'}, 'longest_unique': {'requiresTorch': False, 'hint': "Для 'abba' левая граница после второго b уже 2. При встрече последнего a её нельзя вернуть к 1: используйте max(left,last_seen[char]+1). Корректность важнее скорости печати."}, 'split': {'requiresTorch': False, 'hint': 'Если близкие записи одного человека/каравана попадают и в train, и в validation, оценка может быть слишком оптимистичной. Разделяйте нужную независимую единицу, а не случайные строки.'}, 'retrieval': {'requiresTorch': False, 'hint': 'Повтор слова не повышает score: используем set. Retrieval выбирает источники; generation формулирует ответ. Даже релевантный найденный текст нужно проверить на достоверность.'}, 'attention': {'requiresTorch': True, 'hint': 'Q задаёт запрос, K участвуют в оценке совместимости, V несут смешиваемое содержимое. Деление на sqrt(d) ограничивает масштаб dot products. Это одна query и одна head, без mask; полноценный Transformer сложнее.'}, 'gradient_descent': {'requiresTorch': False, 'hint': 'Gradient указывает локальное направление увеличения loss; вычитаем его. Learning rate — размер шага. Слишком большой шаг может увеличить loss или вызвать расходимость, даже при правильной формуле.'}}

def _dropout_experiment(namespace, transfer=False):
    """Train candidate-created models. Every displayed point is an observed MSE."""
    import torch
    import torch.nn.functional as F
    seed = 43 if transfer else 41
    generator = torch.Generator().manual_seed(seed)
    x_train = torch.rand((12, 2), generator=generator) * 2 - 1
    x_validation = torch.rand((64, 2), generator=generator) * 2 - 1
    def target(x):
        return 1.4*x[:, :1] - .8*x[:, 1:] + .7*torch.sin(4*x[:, :1])
    y_train = target(x_train) + .35*torch.randn((12, 1), generator=generator)
    y_validation = target(x_validation)
    series = []
    outcomes = []
    initial = None
    for probability in (0., .2, .8):
        torch.manual_seed(seed + 100)
        model = namespace["build_dropout_model"](probability)
        weights = [p.detach().clone() for p in model.parameters()]
        if initial is None:
            initial = weights
        elif len(initial) != len(weights) or any(not torch.equal(a, b) for a, b in zip(initial, weights)):
            raise AssertionError("Для сравнения нужны одинаковые начальные weights при одинаковом seed")
        optimizer = torch.optim.SGD(model.parameters(), lr=.08)
        train_values, validation_values = [], []
        for epoch in range(41):
            if epoch:
                model.train()
                optimizer.zero_grad()
                loss = F.mse_loss(model(x_train), y_train)
                loss.backward()
                optimizer.step()
            a = namespace["evaluate_mse"](model, x_train, y_train)
            b = namespace["evaluate_mse"](model, x_validation, y_validation)
            if not all(isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v) for v in (a, b)):
                raise AssertionError("История эксперимента должна состоять из конечных чисел MSE")
            train_values.append(float(a)); validation_values.append(float(b))
        steps = list(range(41))
        series.extend([{"label": f"p={probability:g} · train", "values": train_values, "steps": steps},
                       {"label": f"p={probability:g} · validation", "values": validation_values, "steps": steps}])
        outcomes.append(f"p={probability:g}: train MSE={train_values[-1]:.4f}; validation MSE={validation_values[-1]:.4f}.")
    return {"kind": "training-curves", "title": "Ваши сети: одинаковые данные и начальные веса",
            "xLabel": "Шаг обучения", "yLabel": "MSE", "series": series,
            "observations": outcomes + ["Train и validation измерены в eval: сравниваются одинаковые режимы без случайной маски.",
                "Это один небольшой синтетический опыт. Для устойчивого вывода повторите с другими seed; лучший p здесь не универсален."],
            "parameters": {"seed": seed, "trainingRows": 12, "validationRows": 64, "epochs": 40,
                           "learningRate": .08, "probabilities": [0, .2, .8],
                           "target": "1.4*x1 - 0.8*x2 + 0.7*sin(4*x1)", "trainNoiseStd": .35,
                           "measurementMode": "eval + no_grad", "source": "candidate functions executed by validator"}}


def validate(exercise_id, namespace, variant="practice"):
    """Evaluate outcomes. Fresh processes prevent cross-attempt Python state."""
    if exercise_id not in EXERCISES:
        return {"passed": False, "feedback": "Неизвестное упражнение.", "results": []}
    transfer = variant == "transfer"
    results = []
    artifact = None

    def check(label, work):
        try:
            work()
            results.append({"name": label, "passed": True, "detail": "Проверено на вычисленном результате."})
        except Exception as exc:
            results.append({"name": label, "passed": False,
                            "detail": f"{type(exc).__name__}: {str(exc)[:450]}"})

    def require(condition, message):
        if not condition:
            raise AssertionError(message)

    def close(actual, expected, message="Число не совпало", atol=1e-5):
        require(isinstance(actual, (int, float)) and not isinstance(actual, bool), "Ожидалось число Python")
        require(math.isfinite(float(actual)) and math.isclose(float(actual), float(expected), rel_tol=1e-4, abs_tol=atol),
                f"{message}: получено {actual}, ожидалось {expected:.6g}")

    def function(name):
        fn = namespace.get(name)
        require(callable(fn), f"Определите функцию {name} с именем из задания")
        return fn

    torch = None
    if EXERCISES[exercise_id]["requiresTorch"]:
        try:
            import torch as torch_module
            torch = torch_module
            torch.set_num_threads(1)
            torch.manual_seed(17 if transfer else 7)
        except ImportError:
            return {"passed": False, "feedback": "В этом интерпретаторе нет PyTorch. Установите зависимости или используйте запасной notebook.", "results": []}

    def tensor_close(actual, expected, message="Tensor отличается", atol=1e-5):
        require(isinstance(actual, torch.Tensor), "Нужен настоящий torch.Tensor")
        require(actual.shape == expected.shape, f"Неверный shape: {tuple(actual.shape)}, нужен {tuple(expected.shape)}")
        require(torch.isfinite(actual).all().item(), "В результате NaN или infinity")
        require(torch.allclose(actual.to(expected.dtype), expected, atol=atol, rtol=1e-4), message)

    if exercise_id == "metrics":
        cases = [([1,0,1,0,1,0],[1,1,0,0,1,0]), ([1,1,0,0],[1,0,0,0]), ([0,0],[0,0]), ([],[])] if not transfer else [([1,1,1,0],[0,0,0,0]), ([1,1,0],[1,1,1]), ([0,0,0],[1,1,1]), ([1,0,1],[1,0,1])]
        for i,(truth,pred) in enumerate(cases):
            def work(truth=truth,pred=pred):
                result = function("classification_metrics")(truth[:],pred[:])
                require(isinstance(result,dict), "Верните dict с четырьмя счётчиками и метриками")
                tp=sum(t==p==1 for t,p in zip(truth,pred)); tn=sum(t==p==0 for t,p in zip(truth,pred))
                fp=sum(t==0 and p==1 for t,p in zip(truth,pred)); fn=sum(t==1 and p==0 for t,p in zip(truth,pred))
                precision=tp/(tp+fp) if tp+fp else 0; recall=tp/(tp+fn) if tp+fn else 0
                expected=dict(tp=tp,fp=fp,fn=fn,tn=tn,precision=precision,recall=recall,
                              f1=2*precision*recall/(precision+recall) if precision+recall else 0,
                              accuracy=(tp+tn)/len(truth) if truth else 0)
                for key,value in expected.items(): close(result.get(key),value,key)
            check(f"Confusion matrix, случай {i+1}",work)
    elif exercise_id == "regularization":
        cases=[([1,2,3],[2,4,6],0),([1,2,3],[2,4,6],14),([0,0],[1,2],0)] if not transfer else [([-2,0,2],[-3,0,3],2),([1,1],[3,5],4),([],[],3)]
        for i,(xs,ys,lam) in enumerate(cases):
            def work(xs=xs,ys=ys,lam=lam):
                denom=sum(x*x for x in xs)+lam
                expected=sum(x*y for x,y in zip(xs,ys))/denom if denom else 0
                close(function("ridge_weight")(xs,ys,lam),expected,"Ridge weight")
            check(f"L2, случай {i+1}",work)
        def negative():
            try: function("ridge_weight")([1],[2],-1)
            except ValueError: return
            raise AssertionError("Отрицательная lambda должна вызывать ValueError")
        check("Неверная lambda",negative)
    elif exercise_id == "array_scan":
        cases=[([7,1,5,3,6,4],5),([5,4,3],0),([],0),([1],0)] if not transfer else [([2,1,2,0,1],1),([3,3,3],0),([1,10,0,2],9)]
        for i,(xs,answer) in enumerate(cases):
            def work(xs=xs,answer=answer):
                original=xs[:]; close(function("best_gain")(xs),answer,"Прибыль"); require(xs==original,"Не меняйте вход")
            check(f"Маршрут цен {i+1}",work)
    elif exercise_id == "tensors":
        cases=[[[1,2],[3,4]],[[0.25,-1.5,3.0]],[]] if not transfer else [[[2],[7],[-3]],[[1,2,3],[4,5,6]],[]]
        for i,rows in enumerate(cases):
            def work(rows=rows):
                actual=function("make_tensor")(rows)
                tensor_close(actual,torch.tensor(rows,dtype=torch.float32))
                require(actual.dtype==torch.float32,"Нужен dtype=torch.float32")
                require(actual.device.type=="cpu","Обязательная задача работает на CPU")
            check(f"Tensor {i+1}",work)
    elif exercise_id == "shapes":
        cases=[[[2.,4.,6.],[1.,1.,4.]],[[1.],[7.],[-2.]],[[0.,2.]]] if not transfer else [[[3.,-3.],[10.,4.],[0.,0.]],[[1.,2.,4.,9.]]]
        for i,rows in enumerate(cases):
            def work(rows=rows):
                x=torch.tensor(rows,requires_grad=True)
                actual=function("row_means")(x)
                tensor_close(actual,x.mean(1))
                require(actual.requires_grad,"Потерян computational graph")
                actual.sum().backward()
                tensor_close(x.grad,torch.full_like(x,1/x.shape[1]),"Неверный gradient по оси")
            check(f"Ось и gradient {i+1}",work)
    elif exercise_id == "autograd":
        cases=[[1.,2.,-1.],[0.,.5],[-2.]] if not transfer else [[-3.,1.5,0.],[2./3.,4.]]
        for i,values in enumerate(cases):
            def work(values=values):
                x=torch.tensor(values,requires_grad=True); expected=3*x.detach()**2-2*x.detach()
                actual=function("polynomial_grad")(x)
                tensor_close(actual,expected,"Проверьте формулу x**3-x**2 и backward")
                require(x.grad is not None,"Gradient не накопился в x.grad")
                tensor_close(x.grad,expected)
            check(f"Обратное распространение {i+1}",work)
    elif exercise_id == "module":
        for batch in ([1,4,2] if not transfer else [3,7]):
            def work(batch=batch):
                model=function("build_model")()
                require(isinstance(model,torch.nn.Module),"Верните nn.Module")
                params=list(model.parameters()); require(bool(params),"Нет зарегистрированных параметров")
                require(any(tuple(p.shape)==(1,2) for p in params),"Нужен Linear weight [1,2]")
                require(any(tuple(p.shape)==(1,) for p in params),"Нужен обучаемый bias [1]")
                x=torch.arange(batch*2,dtype=torch.float32).reshape(batch,2)+0.3
                out=model(x); require(isinstance(out,torch.Tensor) and tuple(out.shape)==(batch,1),"Выход должен иметь shape [batch,1]")
                out.sum().backward()
                require(all(p.grad is not None and torch.isfinite(p.grad).all().item() for p in params if p.requires_grad),"Все обучаемые параметры должны получить gradient")
                weight=next(p for p in params if tuple(p.shape)==(1,2))
                tensor_close(weight.grad,x.sum(0,keepdim=True),"Выход должен использовать linear layer")
            check(f"Module: batch={batch}",work)
    elif exercise_id == "losses":
        logits_cases=[([[2.,-1.],[-2.,3.]],[0,1]),([[5.,0.,-2.],[-1.,3.,1.]],[2,1])] if not transfer else [([[0.,0.,0.],[8.,-3.,1.]],[1,0]),([[-100.,100.]],[0])]
        for i,(rows,targets) in enumerate(logits_cases):
            def work(rows=rows,targets=targets):
                pred=torch.tensor(rows,requires_grad=True); target=torch.tensor(targets)
                expected=torch.nn.functional.cross_entropy(pred,target)
                actual=function("loss_for")("classification",pred,target)
                tensor_close(actual,expected,"Cross-entropy отличается: передавайте raw logits")
                ag=torch.autograd.grad(actual,pred,retain_graph=True)[0]
                eg=torch.autograd.grad(expected,pred)[0]; tensor_close(ag,eg,"Неверный gradient loss")
            check(f"Logits → cross-entropy {i+1}",work)
        def regression():
            pred=torch.tensor([1.,-2.,3.] if not transfer else [-4.,0.,7.],requires_grad=True); target=torch.tensor([2.,0.,1.])
            actual=function("loss_for")("regression",pred,target)
            tensor_close(actual,torch.nn.functional.mse_loss(pred,target),"Ожидалась MEAN squared error")
            require(actual.requires_grad,"Loss должна сохранять gradient")
        check("Regression MSE",regression)
        def unknown():
            try: function("loss_for")("unknown",torch.zeros(1),torch.zeros(1))
            except ValueError: return
            raise AssertionError("Неизвестный kind должен вызывать ValueError")
        check("Неизвестная задача",unknown)
    elif exercise_id == "optimizer":
        for i,(w,b,lr) in enumerate(([(0.,0.,.1),(.5,-1.,.03)] if not transfer else [(-.5,2.,.02),(1.,0.,.04)])):
            def work(w=w,b=b,lr=lr):
                model=torch.nn.Linear(1,1); model.eval()
                with torch.no_grad(): model.weight.fill_(w); model.bias.fill_(b)
                model.weight.grad=torch.full_like(model.weight,13); model.bias.grad=torch.full_like(model.bias,-9)
                x=torch.tensor([[-1.],[1.],[2.]]); y=2*x+1
                error=w*x+b-y; expected_loss=(error**2).mean().item()
                expected_w=w-lr*(2*(error*x).mean().item()); expected_b=b-lr*(2*error.mean().item())
                opt=torch.optim.SGD(model.parameters(),lr=lr)
                actual=function("train_step")(model,x,y,opt)
                close(actual,expected_loss,"Возвращайте loss до step")
                close(model.weight.item(),expected_w,"Weight после шага: проверьте zero_grad/backward/step")
                close(model.bias.item(),expected_b,"Bias после шага")
                require(model.training,"model.train() должен быть вызван")
            check(f"SGD и старые gradients {i+1}",work)
    elif exercise_id == "modes":
        def dropout():
            model=torch.nn.Sequential(torch.nn.Linear(2,2),torch.nn.Dropout(.9))
            with torch.no_grad(): model[0].weight.copy_(torch.eye(2)); model[0].bias.zero_()
            model.train(); x=torch.tensor([[2.,3.],[-1.,4.]] if not transfer else [[7.,1.],[2.,-6.]],requires_grad=True)
            actual=function("predict")(model,x)
            tensor_close(actual,x.detach(),"Dropout всё ещё активен; вызовите eval")
            require(not actual.requires_grad,"Не отключена запись graph: нужен torch.no_grad()")
            require(not model.training,"После predict модель должна быть в eval")
        check("Dropout выключен, graph не записан",dropout)
        def batchnorm():
            model=torch.nn.BatchNorm1d(2); model.train()
            with torch.no_grad(): model.running_mean.copy_(torch.tensor([1.,-2.])); model.running_var.copy_(torch.tensor([4.,9.]))
            x=torch.tensor([[3.,1.]],requires_grad=True)
            expected=(x.detach()-model.running_mean)/torch.sqrt(model.running_var+model.eps)
            actual=function("predict")(model,x)
            tensor_close(actual,expected,"В eval BatchNorm использует running statistics")
            require(not actual.requires_grad,"Нужен no_grad, а не только eval")
        check("BatchNorm: один объект в eval",batchnorm)
    elif exercise_id == "training":
        def fitting():
            slope,bias=(-1.5,.7) if transfer else (2.,1.)
            x=torch.tensor([[-2.],[-1.],[0.],[1.],[2.]])
            xv=torch.tensor([[-1.5],[.5],[1.5]])
            y=slope*x+bias; yv=slope*xv+bias
            result=function("fit_line")(x.clone(),y.clone(),xv.clone(),yv.clone())
            require(isinstance(result,dict),"Верните dict model/train_loss/val_loss")
            model=result.get("model"); require(isinstance(model,torch.nn.Module),"model должен быть nn.Module")
            with torch.no_grad():
                train_loss=torch.nn.functional.mse_loss(model(x),y).item(); val_loss=torch.nn.functional.mse_loss(model(xv),yv).item()
            close(result.get("train_loss"),train_loss,"train_loss не соответствует модели")
            close(result.get("val_loss"),val_loss,"val_loss не соответствует отложенным данным")
            require(train_loss<.02 and val_loss<.03,f"Модель ещё не выучила простую прямую: train={train_loss:.4f}, validation={val_loss:.4f}")
        check("Обучение и честные метрики",fitting)
        def leakage():
            x=torch.tensor([[-2.],[-1.],[0.],[1.],[2.]])
            xv=torch.tensor([[-1.5],[.5]])
            a=function("fit_line")(x.clone(),3*x-.5,xv.clone(),3*xv-.5)
            b=function("fit_line")(x.clone(),3*x-.5,xv.clone(),3*xv+30)
            with torch.no_grad(): pa=a["model"](x); pb=b["model"](x)
            tensor_close(pa,pb,"Изменение validation labels изменило обученную модель: возможен leakage или не сброшен seed",atol=1e-4)
            require(b["val_loss"]>100,"Validation loss должна отразить неверные validation labels")
        check("Validation не участвует в обучении",leakage)
    elif exercise_id == "normalization":
        cases=[[[1.,2.,3.],[10.,20.,30.]],[[5.,5.,5.]],[[1.,-1.,2.,-2.]]] if not transfer else [[[4.,8.],[-2.,7.],[3.,3.]],[[100.,101.,103.]]]
        for i,rows in enumerate(cases):
            def work(rows=rows):
                x=torch.tensor(rows,requires_grad=True)
                actual=function("normalize_rows")(x)
                expected=torch.nn.functional.layer_norm(x,(x.shape[-1],),eps=1e-5)
                tensor_close(actual,expected,"Неверная ось, variance или eps")
                require(actual.requires_grad,"Нормализация должна сохранять graph")
            check(f"LayerNorm, случай {i+1}",work)
        def independent():
            row=torch.tensor([[1.,3.,8.]])
            a=function("normalize_rows")(row); b=function("normalize_rows")(torch.cat([row,torch.tensor([[100.,-100.,0.]])]))
            tensor_close(a,b[:1],"Результат строки зависит от соседей: это не LayerNorm по последней оси")
        check("Независимость от соседей по batch",independent)
        def architecture():
            for p in (0., .2, .8):
                model=function("build_dropout_model")(p)
                require(isinstance(model,torch.nn.Sequential),"Верните nn.Sequential из четырёх описанных слоёв")
                layers=list(model.children())
                require(len(layers)==4 and isinstance(layers[0],torch.nn.Linear) and isinstance(layers[1],torch.nn.ReLU)
                        and isinstance(layers[2],torch.nn.Dropout) and isinstance(layers[3],torch.nn.Linear),
                        "Порядок: Linear(2,16), ReLU(), Dropout(p), Linear(16,1)")
                require((layers[0].in_features,layers[0].out_features,layers[3].in_features,layers[3].out_features)==(2,16,16,1),
                        "Размеры слоёв должны образовать маршрут [batch,2] → [batch,16] → [batch,1]")
                close(layers[2].p,p,"В слой Dropout передан не тот p")
                model.eval(); x=torch.tensor([[.3,-.7],[.2,.9]],requires_grad=True)
                prediction=model(x)
                require(tuple(prediction.shape)==(2,1),"Модель должна возвращать [batch,1]")
                prediction.sum().backward()
                require(all(parameter.grad is not None for parameter in model.parameters()),"Все параметры должны участвовать в graph")
        check("Три вероятности и связанная архитектура",architecture)
        def evaluation():
            class ObserveMode(torch.nn.Module):
                def __init__(self):
                    super().__init__(); self.calls=[]
                def forward(self,x):
                    self.calls.append((self.training,torch.is_grad_enabled()))
                    return x[:,:1]*2+x[:,1:]
            model=ObserveMode(); model.train()
            x=torch.tensor([[1.,2.],[-1.,3.]],requires_grad=True)
            y=torch.tensor([[5.],[0.]])
            actual=function("evaluate_mse")(model,x,y)
            close(actual,1.,"evaluate_mse должна усреднять квадраты ошибок")
            require(model.calls and all(not training and not gradients for training,gradients in model.calls),
                    "При проверке нужны model.eval() и torch.no_grad(); это отдельные операции")
            require(not model.training,"Оставьте model в eval после evaluate_mse")
        check("Реальный вызов оценки в eval без graph",evaluation)
        if all(result["passed"] for result in results):
            def experiment():
                nonlocal artifact
                artifact=_dropout_experiment(namespace,transfer)
            check("Обучение трёх сетей и измеренные кривые",experiment)
    elif exercise_id == "two_sum":
        cases=[([2,7,11,15],9),([3,3],6),([],0),([1,2],8)] if not transfer else [([-2,4,1,8],6),([0,0,1],0),([3],6),([4,1,9,3],12)]
        for i,(nums,target) in enumerate(cases):
            def work(nums=nums,target=target):
                original=nums[:]; actual=function("two_sum")(nums,target)
                require(nums==original,"Не меняйте входной список")
                exists=any(nums[a]+nums[b]==target for a in range(len(nums)) for b in range(a+1,len(nums)))
                if not exists: require(actual==[],"Пары нет: верните []"); return
                require(isinstance(actual,(list,tuple)) and len(actual)==2,"Верните два индекса")
                a,b=actual; require(isinstance(a,int) and isinstance(b,int) and 0<=a<len(nums) and 0<=b<len(nums) and a!=b,"Нужны разные допустимые индексы")
                require(nums[a]+nums[b]==target,"Выбранная пара не даёт target")
            check(f"Hash map, случай {i+1}",work)
    elif exercise_id == "agent_loop":
        cases=[(0,3,5,False),(5,1,2,False),(2,2,7,False),(0,2,3,True)] if not transfer else [(3,-1,8,False),(-3,2,0,False),(1,9,2,True)]
        for i,(start,target,budget,stuck) in enumerate(cases):
            def work(start=start,target=target,budget=budget,stuck=stuck):
                calls=[]; position=start
                def tool(name,args):
                    nonlocal position
                    require(len(calls)<budget,"Превышен budget tool calls")
                    require(name=="step" and isinstance(args,dict),"Неверный tool name/schema")
                    require(args.get("position")==position,"Игнорируется результат прошлого tool call")
                    require(args.get("delta")==(1 if target>position else -1),"Неверное направление")
                    calls.append(args.copy())
                    if not stuck: position+=args["delta"]
                    return {"position":position}
                actual=function("navigate")(tool,start,target,budget)
                expected_calls=0 if start==target else budget if stuck else min(abs(target-start),budget)
                require(len(calls)==expected_calls,"Неверное число реальных tool calls")
                require(actual=={"position":position,"calls":len(calls),"reason":"goal" if position==target else "budget"},"Ответ не соответствует реальным вызовам и stop condition")
            check(f"Tool loop, случай {i+1}",work)
    elif exercise_id == "brackets":
        cases=[("{a:[()]}",True),("([)]",False),("",True),("]",False),("((",False)] if not transfer else [("x{[]}(y)",True),("(()))",False),("[({})]",True),("text",True),("{[}",False)]
        for i,(value,expected) in enumerate(cases):
            def work(value=value,expected=expected): require(function("balanced")(value) is expected,"Неверная вложенность или оставшийся stack")
            check(f"Stack, случай {i+1}",work)
    elif exercise_id == "longest_unique":
        cases=[("abcabcbb",3),("bbbbb",1),("",0),("abba",2),("pwwkew",3)] if not transfer else [("dvdf",3),("караван",3),("abcdef",6),("tmmzuxt",5),(" ",1)]
        for i,(value,expected) in enumerate(cases):
            check(f"Sliding window, случай {i+1}",lambda value=value,expected=expected:close(function("longest_unique")(value),expected,"Длина уникального фрагмента"))
    elif exercise_id == "split":
        cases=[([{"id":"a","group":"x"},{"id":"b","group":"y"},{"id":"c","group":"x"},{"id":"d","group":"z"}],{"y"},{"z"}),([],set(),set())] if not transfer else [([{"id":"a","group":1},{"id":"b","group":2},{"id":"c","group":1}],{1},set()),([{"id":"a","group":"z"}],set(),{"z"})]
        for i,(rows,vg,tg) in enumerate(cases):
            def work(rows=rows,vg=vg,tg=tg):
                import copy
                original=copy.deepcopy(rows); actual=function("split_groups")(rows,vg,tg)
                expected={key:[row for row in rows if ("test" if row["group"] in tg else "validation" if row["group"] in vg else "train")==key] for key in ["train","validation","test"]}
                require(actual==expected,"Группа распределена неверно или потерян порядок")
                require(rows==original,"Входные данные изменены")
            check(f"Group split {i+1}",work)
        def overlap():
            try: function("split_groups")([],{"x"},{"x"})
            except ValueError:return
            raise AssertionError("Пересекающиеся группы нужно отвергнуть")
        check("Запрет пересечения",overlap)
    elif exercise_id == "retrieval":
        docs=[{"id":"b","text":"синий фонарь фонарь"},{"id":"a","text":"синий мост"},{"id":"c","text":"красный фонарь"}]
        cases=[("синий фонарь",2,["b","a"]),("неизвестно",3,[]),("синий",0,[])] if not transfer else [("ФОНАРЬ",5,["b","c"]),("мост",1,["a"]),("",2,[])]
        for i,(query,k,expected) in enumerate(cases):
            def work(query=query,k=k,expected=expected):require(function("retrieve")(query,docs,k)==expected,"Неверный score, tie-break по id или top-k")
            check(f"Retrieval {i+1}",work)
    elif exercise_id == "attention":
        for n,d,v in ([(2,2,2),(3,4,1)] if not transfer else [(4,3,2),(1,5,3)]):
            def work(n=n,d=d,v=v):
                q=torch.linspace(-1,1,d,requires_grad=True); keys=torch.arange(n*d,dtype=torch.float32).reshape(n,d)/5
                values=torch.arange(n*v,dtype=torch.float32).reshape(n,v)+.2
                expected=torch.softmax((keys@q)/math.sqrt(d),dim=0)@values
                actual=function("attend")(q,keys,values)
                tensor_close(actual,expected,"Проверьте sqrt(d), softmax по токенам и смешивание V")
                require(actual.requires_grad,"Attention должна сохранять graph")
            check(f"Attention n={n}, d={d}",work)
    elif exercise_id == "gradient_descent":
        cases=[(0.,[1.,2.],[2.,4.],.1),(2.,[1.,2.],[2.,4.],.3),(7.,[],[],.1)] if not transfer else [(-1.,[-2.,1.],[-4.,2.],.05),(3.,[0.,1.],[1.,1.],0.)]
        for i,(w,xs,ys,lr) in enumerate(cases):
            expected=w-lr*2*sum((w*x-y)*x for x,y in zip(xs,ys))/len(xs) if xs else w
            check(f"Gradient descent {i+1}",lambda w=w,xs=xs,ys=ys,lr=lr,expected=expected:close(function("descend")(w,xs,ys,lr),expected,"Новый weight"))

    passed=bool(results) and all(result["passed"] for result in results)
    if passed:
        feedback="Проверки вычислений пройдены на новых входных данных. Объясните результат своими словами; автоматическая проверка кода не оценивает свободный ответ."
    else:
        failed=next((r for r in results if not r["passed"]),None)
        feedback=(failed["detail"] if failed else "Проверки не найдены.") + " Подсказка: " + EXERCISES[exercise_id]["hint"]
    response={"passed":passed,"feedback":feedback,"results":results}
    if artifact is not None:
        response["artifact"]=artifact
    return response


## 1. Четыре числа у городских ворот

Самостоятельно получить четыре исхода классификации и объяснить знаменатель каждой метрики.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

### 1. Один объект — один из четырёх исходов

Здесь 1 означает положительный класс — случай, который требуется обнаружить. TP: истина 1 и прогноз 1; FN: истина 1, прогноз 0; FP: истина 0, прогноз 1; TN: оба 0. Названия говорят об отношении прогноза к истине, а не о том, нравится ли нам действие. Сначала увеличивается один счётчик на объект, затем из счётчиков вычисляются метрики.

### 2. Как сопоставить два списка

zip(y_true, y_pred) даёт пары элементов с одинаковыми позициями. В for truth, prediction in zip(...) пара распаковывается в два имени. По договорённости списки одинаковой длины и содержат только 0/1. Создайте четыре нулевых счётчика до цикла и обновляйте ровно один из них в каждой итерации.

```python
for truth, prediction in zip([1, 0, 1], [1, 1, 0]):
    print(truth, prediction)
```

### 3. Выберите нужное множество в знаменателе

Precision = TP/(TP+FP): среди предсказанных положительных сколько действительно положительных? Recall = TP/(TP+FN): среди всех истинно положительных сколько обнаружено? Accuracy = (TP+TN)/число объектов. F1 = 2*precision*recall/(precision+recall), гармоническое среднее. Оно уменьшается, если одна из двух долей мала. Отрицательный класс в F1 напрямую не входит.

### 4. Нулевой знаменатель — явное соглашение

Если прогнозов 1 нет, precision нельзя получить обычным делением на ноль. Для этого упражнения соответствующая доля равна 0. То же правило применяется к recall, F1 и accuracy пустого набора. Удобна конструкция value_if_true if condition else value_if_false; обычный if тоже подходит. Возвращайте словарь с указанными в условии ключами.

```python
numerator, denominator = 3, 0
ratio = numerator / denominator if denominator else 0.0
```

### Похожий случай, разобранный до самостоятельной задачи

1. Для истины [1,1,0,0,0] и прогноза [1,0,1,0,0] получаются TP=1, FN=1, FP=1, TN=2.
2. Precision=1/2, recall=1/2, accuracy=3/5, F1=1/2.
3. Добавление сотни правильно отвергнутых отрицательных случаев резко поднимет accuracy, но не изменит precision/recall. Это объясняет, почему одной accuracy бывает недостаточно.

```python
tp, fp, fn, tn = 1, 1, 1, 2
precision = tp/(tp+fp)
recall = tp/(tp+fn)
print(2*precision*recall/(precision+recall))
```

### Самостоятельная задача

Напишите classification_metrics(y_true, y_pred). Верните dict: tp, fp, fn, tn, precision, recall, f1, accuracy. Метка 1 — опасный пропуск, который следует задержать. Если знаменатель равен нулю, метрика равна 0. Для пустого набора accuracy тоже 0. Не меняйте входные списки.

In [ ]:
def classification_metrics(y_true, y_pred):
    # Пример: истина [1, 1, 0], решение [1, 0, 1].
    # Здесь TP=1, FN=1, FP=1, TN=0.
    tp = fp = fn = tn = 0
    for truth, prediction in zip(y_true, y_pred):
        # TODO: увеличьте один из четырёх счётчиков.
        pass
    # TODO: precision = TP/(TP+FP); recall = TP/(TP+FN).
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": 0, "recall": 0, "f1": 0, "accuracy": 0}

print(classification_metrics([1, 1, 0], [1, 0, 1]))


In [ ]:
result = validate('metrics', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Precision и recall поменялись местами** — Проверьте вопрос знаменателя: предсказанные положительные или все настоящие положительные?

**На пустом списке возникает ZeroDivisionError** — Условие заранее определило результат для нулевого знаменателя. Обработайте эту ветку до деления.

**Объяснение для собеседования:** Какую метрику выберете при дорогом пропуске опасного события и почему одной метрики всё равно недостаточно?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def classification_metrics(y_true, y_pred):
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
    tn = sum(t == 0 and p == 0 for t, p in zip(y_true, y_pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision,
                recall=recall, f1=f1,
                accuracy=(tp + tn) / len(y_true) if y_true else 0.0)
print(classification_metrics([1, 1, 0], [1, 0, 1]))

```

</details>

## 2. Вес, которому нельзя расти бесконечно

Вычислить коэффициент простой L2-модели и отличить влияние штрафа от измерения качества.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

**Train, validation и test выполняют разные роли**

Train используют для изменения параметров модели. Validation измеряет поведение на других примерах и помогает выбрать настройки. Test оставляют для итоговой оценки после выбора. Обучение на validation labels переносит сведения об оценке в модель: сравнение перестаёт показывать качество на неизвестных ответах.

```text
Измерения растений A–F используются для обучения, G–H — для выбора настройки, I–J — для итоговой проверки. Повторные измерения одного растения держат в одной части.
```

### 1. Модель, ошибка и штраф — три разных объекта

Модель здесь y_hat=w*x: один вес w, свободного члена нет. Обучающая цель — сумма квадратов ошибок плюс lam*w². Первый член требует приблизить ответы, второй делает большие по модулю веса дороже. lam — неотрицательная настройка силы штрафа; её выбирают по validation, а не по финальному test.

### 2. Почему появляется знаменатель

Производная суммы (w*x-y)² по w равна 2*sum((w*x-y)*x). Производная lam*w² равна 2*lam*w. Приравнивая общую производную нулю, получаем w*(sum(x*x)+lam)=sum(x*y). Поэтому w=sum(x*y)/(sum(x*x)+lam). Эта формула относится именно к указанной модели и SUM-loss; для многомерной модели или MEAN-loss выражение меняется.

### 3. Суммы парных произведений в Python

zip(xs,ys) последовательно даёт пары x,y. Выражение sum(x*y for x,y in zip(xs,ys)) суммирует произведения. Его можно заменить явным циклом и накоплением. Оба входных списка по контракту имеют одинаковую длину. Отрицательную lam отклоняют через raise ValueError(...); если итоговый знаменатель нулевой, условие требует вернуть 0.

```python
numerator = sum(x*y for x,y in zip([1.,3.], [2.,6.]))
denominator = sum(x*x for x in [1.,3.])
print(numerator, denominator)  # 20, 10
```

### 4. Как читать три опыта

При фиксированных xs,ys увеличение lam увеличивает знаменатель и уменьшает |w|. Это не гарантирует уменьшения validation MSE: слишком сильное сжатие делает модель недостаточно гибкой. random.Random(SEED) в готовом коде задаёт повторяемый источник учебного шума; rng.gauss(0,2.5) добавляет шум с заданным разбросом. Меняя seed, вы проверяете другой шумный train, а не выбираете удобный финальный test.

### Похожий случай, разобранный до самостоятельной задачи

1. Для xs=[1,3], ys=[2,6] числитель равен 20, сумма x² равна 10.
2. При lam=0 вес равен 2; при lam=10 вес равен 1. Первый вес точно описывает эти два ответа, второй сильнее сжат штрафом.
3. На отдельной проверочной точке x=2,y=4 прогнозы равны 4 и 2. В этом случае штраф ухудшил прогноз. Следовательно, улучшение не заложено в определение регуляризации.

```python
for lam in [0., 10.]:
    weight = 20/(10+lam)
    error = (weight*2-4)**2
    print(lam, weight, error)
```

### Самостоятельная задача

Для модели y_hat = w*x без свободного члена напишите ridge_weight(xs, ys, lam). Минимизируем SUM((w*x-y)^2) + lam*w^2, поэтому w = SUM(x*y)/(SUM(x*x)+lam). Сравните train и validation MSE для lam=0, 2, 20; затем измените SEED для другого шума train. Если знаменатель 0, верните 0. Отрицательную lam отвергайте ValueError. Валидацию выбирают отдельно от train.

In [ ]:
def ridge_weight(xs, ys, lam):
    # lam — сила L2 regularization; x и y одинаковой длины.
    # TODO: запретите отрицательную lam и вычислите две суммы.
    return 0.0

import random
SEED = 7  # Затем попробуйте 19. Это синтетический шум, не новый test set.
rng = random.Random(SEED)
xs = [1.0, 2.0, 3.0]
ys = [2*x + rng.gauss(0, 2.5) for x in xs]
for lam in [0, 2, 20]:
    w = ridge_weight(xs, ys, lam)
    train_mse = sum((w*x-y)**2 for x, y in zip(xs, ys)) / len(xs)
    validation_mse = sum((w*x - 2*x)**2 for x in [1.5, 2.5, 4]) / 3
    print("seed", SEED, "lambda", lam, "weight", round(w, 3),
          "train MSE", round(train_mse, 3), "validation MSE", round(validation_mse, 3))


In [ ]:
result = validate('regularization', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Регуляризация всегда должна улучшать validation** — Это гипотеза о конкретных данных и модели, а не закон. Измерьте результат при нескольких настройках.

**lam=0 вызывает деление на ноль на пустых/нулевых входах** — Условие отдельно задаёт нулевой ответ для нулевого знаменателя.

**Объяснение для собеседования:** Почему L2 может ухудшить обучающую ошибку и при этом оказаться полезной на новых данных?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def ridge_weight(xs, ys, lam):
    if lam < 0:
        raise ValueError("lam должна быть неотрицательной")
    denominator = sum(x*x for x in xs) + lam
    return sum(x*y for x, y in zip(xs, ys)) / denominator if denominator else 0.0

import random
SEED = 7
rng = random.Random(SEED)
xs = [1., 2., 3.]
ys = [2*x + rng.gauss(0, 2.5) for x in xs]
for lam in [0, 2, 20]:
    w = ridge_weight(xs, ys, lam)
    train = sum((w*x-y)**2 for x,y in zip(xs,ys)) / len(xs)
    val = sum((w*x - 2*x)**2 for x in [1.5, 2.5, 4]) / 3
    print("seed", SEED, "lambda", lam, "weight", round(w, 3),
          "train MSE", round(train, 3), "validation MSE", round(val, 3))

```

</details>

## 3. Цена каравана — первая проба на 15 минут

Спроектировать один проход по временной последовательности и объяснить состояние алгоритма.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

### 1. Порядок времени — часть условия

Покупка должна предшествовать продаже. Нельзя взять глобальный минимум и максимум независимо: минимум может наступить после максимума. На каждом дне достаточно знать наименьшую цену среди уже прошедших дней. Разность текущей цены и этого минимума — лучшая возможная продажа именно сегодня.

### 2. Два числа состояния

lowest хранит наименьшую прошлую цену, best — лучшую неотрицательную прибыль. Начальное best=0 означает право не совершать сделку. lowest можно начать с float('inf'): любая реальная первая цена окажется меньше. min(a,b) выбирает меньшее число, max(a,b) — большее. Сначала оцените продажу, затем добавьте текущую цену к истории.

### 3. Инвариант и стоимость

После обработки первых k дней best содержит лучший допустимый результат внутри этого отрезка, а lowest — его минимальную цену. Это утверждение называется инвариантом: оно объясняет смысл состояния на каждом шаге. Один проход делает O(n) действий. Два счётчика занимают O(1) дополнительной памяти независимо от длины входа.

### 4. Короткие и убывающие последовательности

На пустом списке цикл не выполняется, best остаётся 0. Один день тоже не даёт положительной прибыли. При постоянном падении цен все возможные разности неположительны; max с исходным 0 сохраняет право отказаться от сделки. Входной список не нужно сортировать или изменять.

### Похожий случай, разобранный до самостоятельной задачи

1. Цены [9,4,7,2,6]. После 9 минимум=9, прибыль=0; после 4 минимум=4, прибыль=0.
2. Продажа по 7 даёт 3. День с ценой 2 обновляет минимум, но не стирает лучшую прибыль 3.
3. Продажа по 6 даёт 4, поэтому ответ 4. Минимум 2 находится до этой продажи.

```python
lowest, best = 9, 0
price = 4
best = max(best, price-lowest)
lowest = min(lowest, price)
print(lowest, best)
```

### Самостоятельная задача

Напишите best_gain(prices): наибольшая неотрицательная прибыль от одной покупки и более поздней продажи. Покупка не обязательна. Для пустого списка, одного дня или падающих цен верните 0. Постарайтесь использовать один проход и O(1) дополнительной памяти.

In [ ]:
def best_gain(prices):
    # [7, 1, 5, 3, 6, 4] -> 5: купить за 1, продать за 6.
    # TODO: храните минимальную цену СРЕДИ УЖЕ встреченных.
    return 0

print(best_gain([7, 1, 5, 3, 6, 4]))


In [ ]:
result = validate('array_scan', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Список отсортирован перед вычислением** — Сортировка уничтожила порядок дней. Здесь порядок — существенная часть входных данных.

**На падающих ценах ответ отрицателен** — Сделка необязательна. Сравнивайте кандидата с исходной прибылью 0.

**Объяснение для собеседования:** Какое состояние позволяет отказаться от двойного цикла и почему оно достаточно?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def best_gain(prices):
    lowest = float("inf")
    best = 0
    for price in prices:
        best = max(best, price - lowest)
        lowest = min(lowest, price)
    return best
print(best_gain([7, 1, 5, 3, 6, 4]))

```

</details>

## 4. Слитки чисел

Создать настоящий CPU tensor, проверить его форму, числовой тип и связь с исходными данными.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

### 1. Почему обычного списка недостаточно

Список Python хранит объекты и удобен для общей логики. Tensor хранит числовой массив с формой, типом и операциями PyTorch. Возврат исходного списка оставляет данные в другом представлении: у него нет tensor.shape, tensor.dtype и механизмов autograd. NumPy array тоже является массивом, но сам по себе не записывает граф PyTorch.

### 2. Конструктор и именованные аргументы

torch.tensor(data, dtype=..., device=...) создаёт tensor из чисел. dtype и device — именованные аргументы: имя явно сообщает назначение значения. torch.float32 задаёт дробный тип, device='cpu' — процессор. Если передать только целые числа без dtype, автоматически может получиться int64, что отличается от условия.

```python
import torch
vector = torch.tensor([2, 5, 8], dtype=torch.float32, device='cpu')
print(vector.shape, vector.dtype, vector.device)
```

### 3. Как читать форму

У списка [2,5,8] одна ось длины 3: shape [3]. У [[2,5],[8,1]] две оси: shape [2,2]. Каждая внутренняя строка входа имеет одинаковую длину — это заранее заданный контракт прямоугольного входа. Пустой список [] создаёт одномерный tensor с shape [0], а не матрицу [0,0].

### 4. Создание и изменение входа

Функция должна возвращать tensor с теми же значениями, не меняя исходный список. torch.tensor копирует значения из списка. Проверять float значения нужно с учётом ограниченной точности представления; проверка игры использует численный допуск. Сейчас requires_grad включать не требуется: граф появится в следующем упражнении.

### Похожий случай, разобранный до самостоятельной задачи

1. Берём две записи, у каждой по три признака: [[1,4,2],[3,0,5]].
2. Числа должны стать float32 на cpu; размер batch=2, число features=3.
3. Чтение результата.shape подтверждает [2,3]; возврат списка вместо tensor не выполняет договорённость.

```python
import torch
records = [[1,4,2],[3,0,5]]
batch = torch.tensor(records, dtype=torch.float32, device='cpu')
print(batch)
print(batch.shape)
```

### Самостоятельная задача

Напишите make_tensor(rows): из вложенного прямоугольного списка создайте настоящий CPU torch.Tensor с dtype=torch.float32. Сохраните значения и форму. Список [] допустим: shape будет (0,). Изучите shape, dtype и device, затем измените входные числа.

In [ ]:
import torch

def make_tensor(rows):
    # TODO: torch.tensor(..., dtype=..., device=...)
    return rows

result = make_tensor([[1, 2], [3, 4]])
print(result)
print("shape:", getattr(result, "shape", "пока это список"))


In [ ]:
result = validate('tensors', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Числа правильные, но тип int64** — Явно задайте dtype=torch.float32: значения и способ их хранения — разные части контракта.

**Выбран CUDA device** — Этот небольшой опыт рассчитан на CPU и не требует видеокарты.

**Объяснение для собеседования:** Какие три свойства tensor вы проверите перед тем, как передать данные модели?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
def make_tensor(rows):
    return torch.tensor(rows, dtype=torch.float32, device="cpu")
result = make_tensor([[1, 2], [3, 4]])
print(result, result.shape, result.dtype, result.device)

```

</details>

## 5. Провода и оси

Связать ось tensor с её смыслом и усреднить признаки каждого отдельного примера.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

### 1. Ось выбирает, что исчезнет после среднего

В x со shape [batch,features] dim=0 проходит между примерами, dim=1 — между признаками внутри одного примера. x.mean(dim=1) усредняет последнюю группу чисел каждой строки и убирает эту ось. Результат имеет shape [batch]. Если добавить keepdim=True, останется [batch,1], что сейчас не соответствует условию.

### 2. Два возможных ответа — разный смысл

Для [[2,6],[8,12]] среднее строки равно [4,10]. Среднее столбца равно [5,9]. Оба вычисления математически корректны, но отвечают на разные вопросы. Размер результата следует предсказывать по смыслу данных до запуска, а не подбирать dim случайно.

```python
import torch
x = torch.tensor([[2.,6.],[8.,12.]])
print(x.mean(dim=0))  # [5, 9]
print(x.mean(dim=1))  # [4, 10]
```

### 3. Сохранить граф и тип

Операция mean над tensor остаётся частью графа вычислений, если исходный x требует градиентов. Превращение x в список и создание нового tensor разрывает этот путь. Возвращайте результат tensor-операции напрямую, сохранив dtype входа. В самостоятельной задаче проверяются не только числа, но и возможность дальнейшего обучения.

### 4. Связь с другими операциями над формой

reshape меняет разбиение тех же элементов по осям; число элементов сохраняется. Оно не выбирает, что усреднять. При матричном умножении [batch,features] @ [features,outputs] получается [batch,outputs]. Broadcasting расширяет совместимые оси размера 1, но может незаметно создать неправильную пару измерений, если не смотреть на shape.

### Похожий случай, разобранный до самостоятельной задачи

1. Матрица [[1,3,8],[2,2,5]] имеет shape [2,3].
2. Для первой записи среднее 12/3=4, для второй 9/3=3; нужна форма [2].
3. После sum результата backward должен доставить градиенты ко всем шести исходным числам.

```python
import torch
a = torch.tensor([[1.,3.,8.],[2.,2.,5.]], requires_grad=True)
means = a.mean(dim=1)
means.sum().backward()
print(means, a.grad)
```

### Самостоятельная задача

Напишите row_means(x), где x — двумерный float tensor [batch, features]. Верните среднее каждой строки с shape [batch], сохранив dtype и возможность передачи gradient. Проверьте, почему dim=0 даёт другой смысл.

In [ ]:
import torch
def row_means(x):
    # [[2, 4, 6], [1, 1, 4]] -> tensor([4., 2.])
    # TODO: усредните features, то есть dim=1.
    return x
print(row_means(torch.tensor([[2., 4., 6.], [1., 1., 4.]])))


In [ ]:
result = validate('shapes', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Результат имеет длину features** — Вы усреднили по batch: проверьте dim=0 против dim=1.

**Значения верны, но requires_grad=False** — Проверьте, не создавался ли новый tensor из list/NumPy вместо операции над исходным x.

**Объяснение для собеседования:** Как объяснить ошибку оси по смыслу данных, не запоминая конкретные размеры матрицы?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
def row_means(x):
    return x.mean(dim=1)
print(row_means(torch.tensor([[2., 4., 6.], [1., 1., 4.]])))

```

</details>

## 6. След изменения

Проследить вычисление производной через настоящий граф PyTorch и увидеть накопление в leaf tensor.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

### 1. Что именно требуется продифференцировать

Самостоятельная функция вычисляет сумму x³−x² по всем элементам x. ** означает степень, .sum() превращает несколько результатов в одно число. Для отдельного x производная равна 3*x²−2*x: правило для x^n даёт n*x^(n−1), а производные слагаемых складываются с их знаками.

### 2. Знак градиента связан с местным изменением

Если производная положительна, небольшое увеличение входа увеличивает результат; если отрицательна — уменьшает. При x=1 производная x³−x² равна 1; при x=−1 равна 5; при x=0.5 равна −0.25. Это локальное утверждение: оно не обещает монотонности на большом интервале.

### 3. Backward и .grad — разные действия

y.backward() запускает обратный проход. x.grad затем содержит накопленный результат для leaf x. Возврат формулы вручную не показывает работу autograd и не заполняет x.grad. В этой задаче вход свежий и .grad ещё пуст. Повторный обратный проход накапливает градиенты, поэтому при обучении их очищают перед новым шагом.

### 4. Почему не нужен отдельный gradient для каждого элемента

Сумма выходов — scalar, поэтому backward знает начальную производную 1. У каждой независимой компоненты x остаётся собственная производная. Для векторного выхода без sum потребовалось бы дополнительно задать, какой вклад выходных компонент распространяется назад; в этом упражнении это не требуется.

### Похожий случай, разобранный до самостоятельной задачи

1. В аналогичном примере y=sum(x²+3*x). Производная каждой компоненты равна 2*x+3.
2. Для x=[−2,4] ожидаются [−1,11].
3. Строим y tensor-операциями, вызываем backward, затем читаем x.grad; оно совпадает с ручным ожиданием.

```python
import torch
x = torch.tensor([-2.,4.], requires_grad=True)
y = (x**2 + 3*x).sum()
y.backward()
print(x.grad)
```

### Самостоятельная задача

Напишите polynomial_grad(x). x — leaf tensor с requires_grad=True и пустым .grad. Постройте y=SUM(x**3-x**2), вызовите backward и верните x.grad. Не задавайте градиент вручную. Проверка смотрит и возвращённое значение, и накопленный .grad. Объясните знак derivative для разных x.

In [ ]:
import torch
def polynomial_grad(x):
    y = (x**3 - x**2).sum()
    # TODO: запустите обратное распространение от scalar y.
    return x.grad
x = torch.tensor([1., 2., -1.], requires_grad=True)
print(polynomial_grad(x))


In [ ]:
result = validate('autograd', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Вернули правильные числа, но x.grad пуст** — Проверяется фактический обратный проход, а не только формула производной. Вызовите backward на построенном scalar.

**Градиенты неожиданно суммируются** — Обратный проход накапливает .grad. В данном контракте вход свежий; в тренировочном цикле понадобится zero_grad.

**Объяснение для собеседования:** Как отличить вычисленный forward результат от градиента и почему они нужны на разных шагах обучения?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
def polynomial_grad(x):
    y = (x**3 - x**2).sum()
    y.backward()
    return x.grad
x = torch.tensor([1., 2., -1.], requires_grad=True)
print(polynomial_grad(x))

```

</details>

## 7. Маленький искусственный механизм

Собрать собственный nn.Module и объяснить, где хранится его состояние и как проходит вход.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Объект, метод и self**

class описывает объекты со своим состоянием. __init__ вызывается при создании объекта. self — ссылка на конкретный объект; self.value сохраняет поле именно в нём. Метод — функция внутри класса: при вызове counter.add(2) Python передаёт counter как self. Два созданных объекта могут хранить разные значения независимо.

```text
class Counter:
    def __init__(self, start):
        self.value = start
    def add(self, amount):
        self.value += amount
        return self.value
counter = Counter(4)
print(counter.add(2))  # 6
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**nn.Module и зарегистрированные параметры**

nn.Module — базовый класс модели PyTorch. super().__init__() подготавливает механизмы родительского класса. Слой, сохранённый как self.layer, становится частью модели; model.parameters() перечисляет обучаемые параметры. forward описывает вычисление. Вызывать модель следует как model(x): Module вызовет forward и сохранит свои служебные механизмы.

```text
from torch import nn
class Example(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 2)
    def forward(self, x):
        return self.layer(x)
```

### 1. Две функции внутри класса

__init__ создаёт слои один раз, когда вызывают Lantern(). forward используется каждый раз при model(x). Если создавать Linear внутри forward, каждый новый прогноз будет получать новые случайные параметры: обучать устойчивую модель не получится. Сохраняйте слой как поле self.linear в __init__.

### 2. Линейный слой и его размеры

nn.Linear(in_features,out_features) для каждого примера вычисляет W*x+b. При двух входах и одном выходе weight имеет shape [1,2], bias — [1]. Несколько входных строк [batch,2] превращаются в [batch,1]. Bias включён по умолчанию. Это обучаемые параметры, а не сами данные.

```python
from torch import nn
layer = nn.Linear(3,2)
print(layer.weight.shape, layer.bias.shape)  # [2,3], [2]
```

### 3. Регистрация и градиент

После super().__init__() присваивание self.linear=nn.Linear(...) позволяет Module зарегистрировать слой. Тогда model.parameters() найдёт его веса, optimizer сможет менять их, а вычисление self.linear(x) свяжет вход и параметры графом. Возвращение x как есть обходит слой и оставляет параметры бесполезными.

### 4. Фабрика модели

Функция build_model не обучает модель и не возвращает прогноз. Она создаёт и возвращает объект Lantern. Вызывающий код потом передаст ему tensor. Величины случайных начальных весов сейчас не оцениваются: проверяются размеры, зависимость от входа, наличие bias и градиентов.

### Похожий случай, разобранный до самостоятельной задачи

1. Для другой задачи есть три признака и два числовых выхода. Создадим объект с Linear(3,2).
2. Вызов model(batch) применит один и тот же слой к каждой строке batch.
3. Сумма выходов — scalar. После backward у зарегистрированных параметров появляется .grad.

```python
import torch
from torch import nn
class TwoOutputs(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(3,2)
    def forward(self, features):
        return self.projection(features)
example = TwoOutputs()
example(torch.ones(4,3)).sum().backward()
print([p.grad.shape for p in example.parameters()])
```

### Самостоятельная задача

Напишите build_model(), возвращающую nn.Module, который принимает float tensor [batch,2] и выдаёт [batch,1]. Нужен обучаемый linear layer с bias. forward должен зависеть от входа; параметры должны получать gradient. Величины случайных начальных weights не оцениваются.

In [ ]:
import torch
from torch import nn

class Lantern(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.linear = nn.Linear(in_features, out_features)
    def forward(self, x):
        # TODO: используйте свой слой.
        return x

def build_model():
    return Lantern()

print(build_model()(torch.tensor([[1., 2.], [3., 4.]])))


In [ ]:
result = validate('module', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**AttributeError при создании слоя** — В наследнике Module сначала вызовите super().__init__(), затем сохраняйте слои.

**Функция вернула Tensor вместо модели** — build_model — фабрика объекта. Прогноз появляется позже, при вызове возвращённой модели.

**Объяснение для собеседования:** Зачем разделять создание слоёв и forward и что nn.Module добавляет к обычному Python-классу?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
from torch import nn
class Lantern(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)
    def forward(self, x):
        return self.linear(x)
def build_model():
    return Lantern()
print(build_model()(torch.tensor([[1., 2.], [3., 4.]])))

```

</details>

## 8. Что именно считать ошибкой

Выбрать вычисление ошибки по смыслу выхода модели, сохранив граф для обучения.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

### 1. Регрессия: сравнение чисел

Для kind='regression' predictions и targets имеют совместимую одинаковую форму. F.mse_loss(predictions,targets) по умолчанию усредняет квадраты разностей и возвращает scalar tensor. Здесь нужен именно tensor: вызов .item() вернул бы число Python и лишил последующий код возможности вызвать backward.

### 2. Классификация: logits и индексы ответов

Logits — произвольные числовые оценки классов до перевода в вероятности. Для batch из двух объектов и трёх классов predictions имеет shape [2,3]. targets хранит индексы правильных классов, например [2,0], dtype int64. F.cross_entropy сама использует устойчивый вариант log-softmax и отрицательного log правильного класса. Предварительный softmax изменит входную задачу и не нужен.

### 3. Почему уверенная ошибка дороже

Если правильному классу после softmax соответствует вероятность 0.8, вклад cross-entropy равен −ln(0.8); при 0.05 он намного больше. Модель должна увеличить относительную оценку правильного класса. В многоклассовом случае независимые sigmoid для каждого класса описывали бы другую постановку.

### 4. Контрастивная потеря — отдельная цель

Contrastive loss сравнивает представления: близость связанных примеров и различие несвязанных. Это не то же самое, что MSE числового ответа или cross-entropy метки класса. В упражнении реализуются только две прямо названные ветки; для неизвестного kind требуется raise ValueError, чтобы не вычислить молча неправильную задачу.

```python
if kind == 'known':
    result = 1
else:
    raise ValueError('Неизвестный вид задачи')
```

### Похожий случай, разобранный до самостоятельной задачи

1. У одного объекта logits=[0,2,−1], правильный класс имеет индекс 1.
2. Подаём исходные logits и tensor([1],dtype=torch.long) в cross_entropy. Для другого примера регрессии прогнозы [2,6], ответы [4,3] дают MSE 6.5.
3. Обе функции вернут scalar tensors. Градиент останется доступен для обучения модели.

```python
import torch
import torch.nn.functional as F
logits = torch.tensor([[0.,2.,-1.]], requires_grad=True)
labels = torch.tensor([1], dtype=torch.long)
loss = F.cross_entropy(logits, labels)
loss.backward()
print(loss.item(), logits.grad)
```

### Самостоятельная задача

Напишите loss_for(kind, predictions, targets). kind='classification': cross-entropy для raw logits [batch,classes] и индексов классов int64. kind='regression': mean squared error. Верните scalar tensor с gradient. Не делайте softmax перед cross_entropy. Для неизвестного kind выбросите ValueError.

In [ ]:
import torch
import torch.nn.functional as F
def loss_for(kind, predictions, targets):
    if kind == "classification":
        # TODO: F.cross_entropy получает logits.
        return predictions.sum() * 0
    if kind == "regression":
        # TODO: F.mse_loss
        return predictions.sum() * 0
    raise ValueError("Неизвестный kind")
logits = torch.tensor([[2., -1.], [-2., 3.]], requires_grad=True)
print(loss_for("classification", logits, torch.tensor([0, 1])))


In [ ]:
result = validate('losses', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Cross-entropy принимает уже softmax вероятности** — Передайте raw logits. Внутренняя операция cross_entropy сама выполняет необходимое устойчивое преобразование.

**Возвращается float** — Для обучения нужен scalar tensor с графом. .item() уместен для отчёта, но не как возвращаемая loss этой функции.

**Объяснение для собеседования:** Чем отличаются числовой прогноз, logits, вероятности и функция потерь?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
import torch.nn.functional as F
def loss_for(kind, predictions, targets):
    if kind == "classification":
        return F.cross_entropy(predictions, targets)
    if kind == "regression":
        return F.mse_loss(predictions, targets)
    raise ValueError("Неизвестный kind")
logits = torch.tensor([[2., -1.], [-2., 3.]], requires_grad=True)
print(loss_for("classification", logits, torch.tensor([0, 1])))

```

</details>

## 9. Один честный шаг обучения

Выполнить один настоящий шаг SGD и различить создание градиентов и изменение весов.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**nn.Module и зарегистрированные параметры**

nn.Module — базовый класс модели PyTorch. super().__init__() подготавливает механизмы родительского класса. Слой, сохранённый как self.layer, становится частью модели; model.parameters() перечисляет обучаемые параметры. forward описывает вычисление. Вызывать модель следует как model(x): Module вызовет forward и сохранит свои служебные механизмы.

```text
from torch import nn
class Example(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 2)
    def forward(self, x):
        return self.layer(x)
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

### 1. Роли четырёх операций

optimizer.zero_grad() очищает старые градиенты. model(x) и MSE вычисляют текущую ошибку. loss.backward() получает новые градиенты по текущим параметрам. optimizer.step() использует их для обновления параметров. В SGD изменение имеет вид weight ← weight − learning_rate*gradient. Перестановка этих действий меняет смысл процедуры.

### 2. Почему очистка нужна каждый раз

PyTorch накапливает градиенты при backward. Это полезно в некоторых алгоритмах, но здесь каждый train_step должен использовать только текущий batch. Проверка намеренно оставляет старые градиенты: корректная функция очистит их перед новым обратным проходом. model.train() переключает режим слоёв, но не очищает .grad.

### 3. Переданный optimizer уже связан с моделью

Вызывающий код создаёт SGD(model.parameters(),lr=...). Не нужно создавать новый optimizer внутри train_step: это скроет переданную настройку и для других оптимизаторов потеряет накопленное состояние. В этой функции используйте переданный объект и тот же model.

### 4. Что именно вернуть

loss была вычислена до обновления параметров и содержит числовую ошибку того forward прохода. loss.item() извлекает это scalar значение как float Python для журнала. Повторно вычисленная ошибка после optimizer.step могла бы быть другой; условие явно просит величину до шага.

### Похожий случай, разобранный до самостоятельной задачи

1. Для одного веса w=0, входа x=2 и цели y=6 прогноз 0, MSE 36.
2. Производная по w равна 2*(w*x−y)*x = −24. При learning rate 0.05 новый вес равен 1.2.
3. Ошибка, возвращённая функцией шага, остаётся 36: она относится к прогнозу до изменения веса.

```python
import torch
w = torch.nn.Parameter(torch.tensor(0.))
optimizer = torch.optim.SGD([w], lr=.05)
optimizer.zero_grad()
loss = (w*2-6)**2
loss.backward()
optimizer.step()
print(w.item(), loss.item())
```

### Самостоятельная задача

Напишите train_step(model,x,y,optimizer): переведите model в train, очистите старые gradients, вычислите MSE, выполните backward и optimizer.step. Верните float значения loss ДО изменения параметров. Проверка подмешивает старый gradient, чтобы обнаружить пропущенный zero_grad.

In [ ]:
import torch
import torch.nn.functional as F
def train_step(model, x, y, optimizer):
    model.train()
    # TODO: очистка gradients
    loss = F.mse_loss(model(x), y)
    # TODO: backward, затем шаг optimizer
    return loss.item()

model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
x = torch.tensor([[1.], [2.], [3.]])
y = 2*x + 1
for step in range(3):
    print("step", step, "loss", train_step(model, x, y, optimizer))


In [ ]:
result = validate('optimizer', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Веса меняются слишком сильно при повторном запуске** — Проверьте, очищаются ли градиенты перед новым backward.

**Backward вызван, но веса прежние** — Backward вычисляет производные; изменение выполняет optimizer.step().

**Объяснение для собеседования:** Какая ошибка возникает при пропуске zero_grad и почему model.train() её не исправляет?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
import torch.nn.functional as F
def train_step(model, x, y, optimizer):
    model.train()
    optimizer.zero_grad()
    loss = F.mse_loss(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.item()
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
x = torch.tensor([[1.], [2.], [3.]])
y = 2*x + 1
for step in range(3):
    print("step", step, "loss", train_step(model, x, y, optimizer))

```

</details>

## 10. Проверка без случайной завесы

Провести оценку модели в правильном режиме, независимо управляя слоями и записью графа.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**nn.Module и зарегистрированные параметры**

nn.Module — базовый класс модели PyTorch. super().__init__() подготавливает механизмы родительского класса. Слой, сохранённый как self.layer, становится частью модели; model.parameters() перечисляет обучаемые параметры. forward описывает вычисление. Вызывать модель следует как model(x): Module вызовет forward и сохранит свои служебные механизмы.

```text
from torch import nn
class Example(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 2)
    def forward(self, x):
        return self.layer(x)
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

### 1. Train/eval меняет поведение некоторых слоёв

model.train() включает режим обучения, model.eval() — режим оценки. Dropout в train случайно выключает активации, а в eval пропускает их без случайного обнуления. BatchNorm в обычной конфигурации при train использует текущий batch и обновляет running statistics, а в eval использует накопленные. Сами веса model.eval() не обучает и не замораживает.

### 2. No_grad отвечает на другой вопрос

with torch.no_grad(): временно отключает запись операций для обратного прохода. Внутри этого блока вычисляется прогноз, который не требует хранения графа. Это экономит память при оценке. Однако no_grad не переключает Dropout/BatchNorm: случайная маска осталась бы включена, если забыть eval.

### 3. Возвращаемый объект и состояние

Функция predict должна вернуть tensor результата model(x), а модель после вызова оставить в eval. Это контракт данной задачи, а не обязанность любой predict-функции в мире. При следующем обучении вызывающий код должен снова сделать train(). Возвращение tensor.detach() после обычного прохода не показывает требуемый режим no_grad во время вычисления.

### 4. Как читать контекстный блок

with задаёт область действия контекстного менеджера. После выхода из torch.no_grad() прежний режим градиентов восстанавливается. Слои модели при этом сами в train не возвращаются: их режим переключён отдельно. Отступ объединяет все операции, которые должны выполняться без записи графа.

```python
with torch.no_grad():
    prediction = model(features)
# Здесь запись графа снова определяется внешним режимом.
```

### Похожий случай, разобранный до самостоятельной задачи

1. Берём слой, содержащий Dropout с большой вероятностью. В train два прохода могут отличаться даже на одинаковом входе.
2. После eval случайное обнуление выключается, но параметры Linear всё ещё могут записываться в граф.
3. Помещаем forward в no_grad и проверяем result.requires_grad: для оценки он False.

```python
import torch
model = torch.nn.Sequential(torch.nn.Linear(3,2), torch.nn.Dropout(.5))
x = torch.ones(2,3, requires_grad=True)
model.eval()
with torch.no_grad():
    result = model(x)
print(result.requires_grad, model.training)
```

### Самостоятельная задача

Напишите predict(model,x): установите eval mode и вычислите результат внутри torch.no_grad(). Верните tensor. Состояние model оставьте eval. Разберитесь: eval изменяет поведение Dropout/BatchNorm, no_grad отключает запись computational graph. Это две разные операции.

In [ ]:
import torch
def predict(model, x):
    # TODO: eval + контекст no_grad, затем model(x).
    return model(x)

model = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Dropout(0.8))
x = torch.tensor([[1., 2.]], requires_grad=True)
print(predict(model, x))


In [ ]:
result = validate('modes', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Прогноз случайно меняется при оценке** — Один no_grad не выключает Dropout. Переключите модель в eval.

**Eval установлен, но требуется gradient** — Eval и запись графа независимы. Выполните forward внутри no_grad.

**Объяснение для собеседования:** Почему для обычной проверки модели одновременно нужны eval() и no_grad()?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
def predict(model, x):
    model.eval()
    with torch.no_grad():
        return model(x)
model = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Dropout(0.8))
x = torch.tensor([[1., 2.]], requires_grad=True)
print(predict(model, x))

```

</details>

## 11. Опыт, который переживает новые данные

Соединить модель, loss, optimizer и независимую оценку в воспроизводимый тренировочный цикл.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

**nn.Module и зарегистрированные параметры**

nn.Module — базовый класс модели PyTorch. super().__init__() подготавливает механизмы родительского класса. Слой, сохранённый как self.layer, становится частью модели; model.parameters() перечисляет обучаемые параметры. forward описывает вычисление. Вызывать модель следует как model(x): Module вызовет forward и сохранит свои служебные механизмы.

```text
from torch import nn
class Example(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 2)
    def forward(self, x):
        return self.layer(x)
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

**Train, validation и test выполняют разные роли**

Train используют для изменения параметров модели. Validation измеряет поведение на других примерах и помогает выбрать настройки. Test оставляют для итоговой оценки после выбора. Обучение на validation labels переносит сведения об оценке в модель: сравнение перестаёт показывать качество на неизвестных ответах.

```text
Измерения растений A–F используются для обучения, G–H — для выбора настройки, I–J — для итоговой проверки. Повторные измерения одного растения держат в одной части.
```

### 1. Что в задаче уже задано

Входы — четыре float tensors. x_train и y_train относятся к обучению, x_validation и y_validation — к оценке. Каждый tensor имеет форму [n,1]. Модель Linear(1,1) содержит один вес и bias: y_hat=w*x+b. Она подходит для данной маленькой линейной закономерности; это не обещание пригодности для произвольных данных.

### 2. Повторяемый старт

torch.manual_seed(7) задаёт генератор PyTorch перед созданием модели, поэтому одинаковые вызовы начнут с одинаковых случайных весов. Создайте Linear и SGD один раз до цикла. range(250) выполняет 250 шагов; _ означает, что номер шага здесь не нужен. Внутри шага используйте только train: zero_grad → MSE(model(x_train),y_train) → backward → step.

### 3. Оценка после обучения

После цикла включите eval. В блоке no_grad вычислите отдельно MSE на train и на validation. .item() превратит scalar tensors в числа для отчёта. Возвращаемый словарь должен содержать именно обученный объект model и его реальные train_loss/val_loss. Числа не следует подставлять вручную или брать от другой модели.

### 4. Как обнаружить скрытое обучение на validation

Представьте два запуска с одинаковым train, seed и x_validation, но разными y_validation. Обученные параметры должны совпасть; только измеренная validation loss изменится. Если параметры изменились, оценочные ответы повлияли на обучение. Игра проверяет это поведение непосредственно.

### 5. Границы результата

Маленькая учебная прямая позволяет быстро увидеть снижение ошибки на CPU. Число шагов и learning rate — гиперпараметры, которые можно исследовать. Низкая ошибка на этом наборе подтверждает процедуру для этого случая; для новой постановки понадобятся подходящая модель, независимые данные и повторения.

### Похожий случай, разобранный до самостоятельной задачи

1. Пусть новые учебные данные соответствуют y=−0.5*x+2, а начальная сеть этого не знает.
2. Каждый шаг оценивает ошибку только по train и меняет вес/bias в направлении её уменьшения.
3. Проверочные x лежат между обучающими точками. Их ответы используются после обучения только для вычисления val_loss.

```python
import torch
import torch.nn.functional as F
torch.manual_seed(11)
net = torch.nn.Linear(1,1)
opt = torch.optim.SGD(net.parameters(), lr=.1)
x = torch.tensor([[-1.],[0.],[1.]])
y = -.5*x+2
for epoch in range(20):
    net.train()
    opt.zero_grad()
    error = F.mse_loss(net(x), y)
    error.backward()
    opt.step()
net.eval()
with torch.no_grad():
    print(F.mse_loss(net(x), y).item())
```

### Самостоятельная задача

Напишите fit_line(x_train,y_train,x_validation,y_validation). Данные — float tensors [n,1]. Обучите nn.Linear(1,1) только на train, затем верните dict {model,train_loss,val_loss}; обе loss — float MSE. Сбросьте seed в функции. Validation labels участвуют только в измерении. Меняйте число шагов и learning rate и объясняйте изменения.

In [ ]:
import torch
import torch.nn.functional as F
def fit_line(x_train, y_train, x_validation, y_validation):
    torch.manual_seed(7)
    model = torch.nn.Linear(1, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.08)
    for _ in range(250):
        # TODO: train → zero_grad → loss на TRAIN → backward → step
        pass
    model.eval()
    with torch.no_grad():
        train_loss = F.mse_loss(model(x_train), y_train).item()
        val_loss = F.mse_loss(model(x_validation), y_validation).item()
    return {"model": model, "train_loss": train_loss, "val_loss": val_loss}

x = torch.tensor([[-2.], [-1.], [0.], [1.], [2.]])
xv = torch.tensor([[-1.5], [0.5], [1.5]])
result = fit_line(x, 2*x+1, xv, 2*xv+1)
print("train:", result["train_loss"], "validation:", result["val_loss"])


In [ ]:
result = validate('training', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Validation loss хорошая, но веса меняются при смене её ответов** — Валидационные метки попали в обучение. Проверьте переменные в loss внутри цикла.

**Результат словаря не совпадает с моделью** — Измерьте обе ошибки после последнего optimizer.step на том же объекте model.

**Объяснение для собеседования:** Как построить проверку, которая отличает честную оценку от подмешивания validation labels в обучение?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
import torch.nn.functional as F
def fit_line(x_train, y_train, x_validation, y_validation):
    torch.manual_seed(7)
    model = torch.nn.Linear(1, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.08)
    model.train()
    for _ in range(250):
        optimizer.zero_grad()
        loss = F.mse_loss(model(x_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        train_loss = F.mse_loss(model(x_train), y_train).item()
        val_loss = F.mse_loss(model(x_validation), y_validation).item()
    return {"model": model, "train_loss": train_loss, "val_loss": val_loss}
x = torch.tensor([[-2.], [-1.], [0.], [1.], [2.]])
xv = torch.tensor([[-1.5], [0.5], [1.5]])
result = fit_line(x, 2*x+1, xv, 2*xv+1)
print("train:", result["train_loss"], "validation:", result["val_loss"])

```

</details>

## 12. Контролируемый опыт: нормализация и Dropout

Провести настоящий контролируемый опыт с Dropout и отличить его от нормализации.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**nn.Module и зарегистрированные параметры**

nn.Module — базовый класс модели PyTorch. super().__init__() подготавливает механизмы родительского класса. Слой, сохранённый как self.layer, становится частью модели; model.parameters() перечисляет обучаемые параметры. forward описывает вычисление. Вызывать модель следует как model(x): Module вызовет forward и сохранит свои служебные механизмы.

```text
from torch import nn
class Example(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 2)
    def forward(self, x):
        return self.layer(x)
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

**Train, validation и test выполняют разные роли**

Train используют для изменения параметров модели. Validation измеряет поведение на других примерах и помогает выбрать настройки. Test оставляют для итоговой оценки после выбора. Обучение на validation labels переносит сведения об оценке в модель: сравнение перестаёт показывать качество на неизвестных ответах.

```text
Измерения растений A–F используются для обучения, G–H — для выбора настройки, I–J — для итоговой проверки. Повторные измерения одного растения держат в одной части.
```

### 1. Два разных способа менять промежуточные значения

LayerNorm вычитает среднее и делит на масштаб внутри каждого примера. Для строки это (x−mean)/sqrt(var+eps), где var — средний квадрат отклонения, eps=1e-5 защищает от деления на ноль. F.layer_norm(x,(x.shape[-1],),eps=1e-5) применяет это к последней оси; scale/bias здесь отсутствуют. x.shape[-1] берёт длину последней оси; запятая создаёт кортеж из одного числа. Строки независимы от соседей по batch.

### 2. BatchNorm использует другой набор статистик

Для типичного BatchNorm1d признаки нормируются с использованием batch. В train слой оценивает текущие статистики и накапливает running values; в eval обычно использует накопленные. LayerNorm по последней оси берёт статистики внутри текущей строки и не переключает их на running batch statistics. Поэтому эти слои не следует заменять друг другом только из-за слова norm.

### 3. Что делает Dropout

Dropout с вероятностью p случайно обнуляет каждое промежуточное значение в режиме train. Сохранённые значения делятся на 1−p, чтобы сохранить их математическое ожидание. При p=0.5 один возможный результат для [2,4,6,8] — [4,0,12,0]; ровно половина не обязана выключаться в каждом проходе. В eval Dropout ничего случайно не выключает. Слишком большое p может мешать модели учиться; улучшение не гарантировано.

### 4. Собрать маршрут данных

nn.Sequential последовательно применяет перечисленные слои. Для задания нужны Linear(2,16), ReLU(), Dropout(p), Linear(16,1). ReLU заменяет отрицательные значения нулём и оставляет положительные. Формы меняются [batch,2] → [batch,16] → [batch,16] → [batch,16] → [batch,1]. Все обучаемые параметры должны оставаться в Module и получать градиенты.

```python
from torch import nn
# Аналог для трёх входов и двух выходов:
example = nn.Sequential(nn.Linear(3,8), nn.ReLU(), nn.Dropout(.3), nn.Linear(8,2))
```

### 5. Проверка без случайной маски

evaluate_mse сначала делает model.eval(), затем внутри with torch.no_grad() вычисляет F.mse_loss(model(x),y).item(). Eval отключает случайную маску и меняет режим BatchNorm; no_grad отключает граф. Оставьте model в eval после функции. Перед следующим шагом обучения стенд явно вызывает train(), чтобы Dropout снова работал.

### 6. Условия настоящего эксперимента

Стенд создаёт 12 обучающих и 64 проверочных примера с двумя числовыми признаками. Цель y=1.4*x1−0.8*x2+0.7*sin(4*x1); в train добавлен фиксированный шум. Формула описывает синтетические данные, биологических знаний для задания не требуется. Для каждого p=0,0.2,0.8 повторно устанавливаются одинаковый seed и начальные веса; используются SGD lr=0.08 и 40 полных шагов по train. Validation не участвует в обновлениях.

### 7. Как читать полученные кривые

Каждый график построен после фактического исполнения ваших функций. И train, и validation MSE измеряются в eval, поэтому сравнение не смешивает разные режимы Dropout. Важны величина и развитие ошибок, а не заранее объявленный победитель. Один seed даёт один опыт: для устойчивого вывода нужны повторения. Итоговый test не используется для выбора p.

### Похожий случай, разобранный до самостоятельной задачи

1. Сначала нормируем строки [[1,3],[20,24]]: обе дают примерно [−1,1], хотя масштабы и средние различны.
2. Затем берём отдельную сеть Linear(3,8) → ReLU → Dropout(0.3) → Linear(8,2), чтобы увидеть, что нормализация и случайное обнуление — разные операции.
3. Оценка в eval внутри no_grad возвращает одно число MSE. Перед обучением необходимо снова включить train.

```python
import torch
import torch.nn.functional as F
from torch import nn
rows = torch.tensor([[1.,3.],[20.,24.]])
print(F.layer_norm(rows, (2,), eps=1e-5))
net = nn.Sequential(nn.Linear(3,8), nn.ReLU(), nn.Dropout(.3), nn.Linear(8,2))
net.eval()
with torch.no_grad():
    value = F.mse_loss(net(torch.ones(2,3)), torch.zeros(2,2)).item()
print(value)
```

### Самостоятельная задача

Готовая normalize_rows(x) показывает LayerNorm по последней оси без обучаемых scale/bias, eps=1e-5; её можно исследовать. Допишите build_dropout_model(p): CPU nn.Sequential из Linear(2,16), ReLU, Dropout(p), Linear(16,1). Допишите evaluate_mse(model,x,y): переведите модель в eval, измерьте среднюю квадратичную ошибку внутри torch.no_grad() и верните float, оставив модель в eval. Проверка обучит три ваши сети с p=0, 0.2, 0.8 на одинаковом разбиении и из одинаковых начальных весов. Графики показывают настоящие измерения; улучшение от Dropout не гарантируется. Перед каждым шагом обучения учебный стенд сам вызывает train().

In [ ]:
import torch
import torch.nn.functional as F
def normalize_rows(x):
    return F.layer_norm(x, (x.shape[-1],), eps=1e-5)

def build_dropout_model(p):
    # Порядок слоёв: Linear(2,16) → ReLU → Dropout(p) → Linear(16,1).
    # TODO: верните torch.nn.Sequential с этими слоями.
    return torch.nn.Identity()

def evaluate_mse(model, x, y):
    # TODO: eval; with torch.no_grad(); F.mse_loss(...).item().
    return 0.0

print(normalize_rows(torch.tensor([[1., 2., 3.], [10., 20., 30.]])))
# После исправления функций кнопка «Запустить» обучит три сети
# на учебных данных и покажет графики train/validation.


In [ ]:
result = validate('normalization', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Сравнение запусков меняет сразу и p, и данные** — Для причинного сравнения меняют одну выбранную настройку; остальные условия удерживают одинаковыми.

**При validation остаётся случайный Dropout** — No_grad не меняет режим слоя. Перед измерением явно вызовите eval.

**При p=0.8 качество хуже** — Это допустимый результат опыта. Сильная регуляризация может помешать обучению; оценка не обязана улучшиться.

**Объяснение для собеседования:** Как организовать эксперимент, который проверяет влияние Dropout, и какие выводы нельзя сделать по одному seed?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
import torch.nn.functional as F
def normalize_rows(x):
    return F.layer_norm(x, (x.shape[-1],), eps=1e-5)
def build_dropout_model(p):
    return torch.nn.Sequential(torch.nn.Linear(2, 16), torch.nn.ReLU(),
                               torch.nn.Dropout(p), torch.nn.Linear(16, 1))
def evaluate_mse(model, x, y):
    model.eval()
    with torch.no_grad():
        return F.mse_loss(model(x), y).item()
print(normalize_rows(torch.tensor([[1., 2., 3.], [10., 20., 30.]])))

```

</details>

## 13. Два груза — вторая проба на 15 минут

Использовать словарь как память о предыдущих индексах и сохранить ограничение разных элементов.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

### 1. Из пары — в поиск дополнения

Если текущее число value, нужный партнёр равен target−value. Достаточно узнать, встречался ли он раньше. Словарь seen хранит число → один ранее встреченный индекс. Это именно индексы: возвращение самих чисел нарушает контракт, даже если их сумма правильная.

### 2. Enumerate связывает индекс и значение

for i,value in enumerate(nums) даёт пары (0,nums[0]), (1,nums[1]) и так далее. До обработки текущего элемента в seen должны быть только прошлые элементы. Сначала проверяется дополнение, потом выполняется seen[value]=i. Такой порядок автоматически исключает использование текущего индекса дважды.

### 3. Повторяющиеся числа допустимы

В [4,4] при target=8 первый элемент запоминается, а второй находит его. Это два разных индекса. Перезапись более раннего индекса тем же числом не разрушает возможность найти некоторую правильную пару: условие принимает любую. Если цикл закончился без пары, верните [] — включая пустой вход и список из одного элемента.

### 4. Что означает ожидаемая O(n)

Поиск ключа и вставка в обычный dict в среднем требуют постоянного числа действий. Для n элементов получается ожидаемая O(n) работа и O(n) дополнительная память. Это описание структуры данных при нормальной работе хеширования, а не обещание фиксированного времени в миллисекундах.

### Похожий случай, разобранный до самостоятельной задачи

1. Вход [6,1,8,3], target=9. На 6 память пуста: запоминаем {6:0}.
2. На 1 нужно 8, его ещё нет. На 8 нужно 1, которое хранится по индексу 1.
3. Возвращается [1,2]. Индексы различны, значения 1 и 8 действительно дают 9.

```python
seen = {6:0, 1:1}
index, value, target = 2, 8, 9
needed = target-value
if needed in seen:
    print([seen[needed], index])
```

### Самостоятельная задача

Напишите two_sum(nums,target). Верните индексы двух РАЗНЫХ элементов с нужной суммой, либо [] если пары нет. Любая правильная пара подходит; вход не меняйте. Храните ранее увиденные числа в hash map и добейтесь ожидаемого O(n).

In [ ]:
def two_sum(nums, target):
    seen = {}
    for i, value in enumerate(nums):
        # TODO: нужен target-value. Проверьте ДО добавления value.
        pass
    return []
print(two_sum([2, 7, 11, 15], 9))


In [ ]:
result = validate('two_sum', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Для [4] найден ответ [0,0]** — Текущее число добавлено в память до поиска дополнения. Ищите среди прошлых индексов.

**Вместо индексов возвращены значения** — Храните в словаре позиции и верните их; число служит ключом поиска.

**Объяснение для собеседования:** Как порядок проверки и вставки обеспечивает разные индексы без отдельного перебора пар?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def two_sum(nums, target):
    seen = {}
    for i, value in enumerate(nums):
        if target - value in seen:
            return [seen[target - value], i]
        seen[value] = i
    return []
print(two_sum([2, 7, 11, 15], 9))

```

</details>

## 14. Джинн, умеющий остановиться

Реализовать ограниченный цикл инструментальных действий, который подчиняется фактическим наблюдениям.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

### 1. Функция может быть аргументом

Параметр call_tool содержит вызываемую функцию. Вызов call_tool('step',arguments) передаёт имя инструмента и словарь аргументов; результат — новый словарь. Это обычный Python-механизм передачи функций, а не особый синтаксис LLM. В упражнении выбор следующего действия задан правилом: такой test double позволяет проверить цикл без загрузки модели.

### 2. Request и observation не одно и то же

Запрос содержит текущую position и delta, равную +1 или −1. Ответ содержит фактическую position. Инструмент может не выполнить ожидаемое перемещение, поэтому нельзя обновлять позицию собственным position+=delta. Прочитайте result['position'] и используйте именно её на следующем шаге.

### 3. Две независимые причины остановки

Цель достигнута, когда position==target. Бюджет закончился, когда calls>=budget. while position != target and calls < budget проверяет обе границы до следующего вызова. После реального вызова увеличивайте calls на 1, даже если позиция не изменилась. budget по контракту — неотрицательное целое число.

### 4. Выбрать направление и объяснить итог

Если target>position, delta=1, иначе −1. При совпадении позиций цикл вообще не должен запускаться. Итоговый dict содержит фактическую position, число реальных calls и reason: 'goal' при достигнутой цели, иначе 'budget'. Такой отчёт различает намерение, исполнение и ограничение ресурса.

### 5. Как это переносится на настоящего агента

В отдельной панели игры выбор действия может делать ответ подключённой локальной модели. Исполнитель всё равно проверяет имя/аргументы, выполняет разрешённый инструмент, передаёт результат и ограничивает число итераций. Успех этого Python-упражнения подтверждает логику цикла на учебных инструментах, а подключение живой модели проверяется отдельно.

### Похожий случай, разобранный до самостоятельной задачи

1. Старт 4, цель 1, budget=2. Первый запрос step(position=4,delta=−1) получает position=3.
2. Второй запрос использует наблюдаемую 3 и получает 2. Бюджет исчерпан: итог position=2,calls=2,reason='budget'.
3. Если инструмент оба раза возвращает прежнюю позицию 4, результат обязан остаться 4, а число вызовов всё равно будет 2.

```python
def observe(name, arguments):
    return {'position': arguments['position']}  # инструмент не сдвинулся
result = observe('step', {'position':4, 'delta':-1})
position = result['position']
print(position)  # 4, а не ожидаемое 3
```

### Самостоятельная задача

Напишите navigate(call_tool,start,target,budget). Пока позиция не равна target и есть budget, вызовите call_tool('step', {'position':position,'delta':±1}) и прочитайте возвращённый {'position':...}. Верните {'position':int,'calls':int,'reason':'goal' либо 'budget'}. Это явно детерминированный test double, не ответ LLM. Нельзя просто присвоить target без tool call.

In [ ]:
def navigate(call_tool, start, target, budget):
    position, calls = start, 0
    # TODO: ограниченный loop, tool request → result → update → stop.
    return {"position": position, "calls": calls,
            "reason": "goal" if position == target else "budget"}

def demo_tool(name, args):
    print("tool request:", name, args)
    return {"position": args["position"] + args["delta"]}
print(navigate(demo_tool, 0, 3, 5))


In [ ]:
result = validate('agent_loop', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Код телепортирует позицию к цели** — Изменение состояния должно быть подтверждено реальным вызовом инструмента и его результатом.

**Застрявший инструмент вызывает вечный цикл** — Бюджет считают по вызовам, а не по успешным перемещениям.

**Объяснение для собеседования:** Какие части агентного цикла должны остаться проверяемыми независимо от качества самой LLM?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def navigate(call_tool, start, target, budget):
    position, calls = start, 0
    while position != target and calls < budget:
        delta = 1 if target > position else -1
        result = call_tool("step", {"position": position, "delta": delta})
        position = result["position"]
        calls += 1
    return {"position": position, "calls": calls,
            "reason": "goal" if position == target else "budget"}
def demo_tool(name, args):
    print("tool request:", name, args)
    return {"position": args["position"] + args["delta"]}
print(navigate(demo_tool, 0, 3, 5))

```

</details>

## 15. Запечатанное письмо — третья проба на 15 минут

Проверить вложенную структуру с помощью стека ожидаемых закрывающих символов.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

### 1. Почему недостаточно посчитать скобки

В '([)]' число открывающих и закрывающих символов каждого типа совпадает, но порядок неверен: первым нужно закрыть самый последний открытый фрагмент. Стек — структура, у которой последним добавленным элементом пользуются первым. В Python его можно хранить как список.

### 2. Операции стека

stack.append(value) добавляет значение в конец. stack.pop() удаляет и возвращает последний элемент; у пустого списка это вызвало бы ошибку. Поэтому сначала проверяйте пустоту: not stack истинно для []. В нашем варианте в стеке хранятся ожидаемые закрывающие символы, например после '(' кладётся ')'.

### 3. Три типа входных символов

Словарь opening связывает '(', '[', '{' с парными закрывающими символами. Если char in opening, добавьте ожидаемую пару. Если char входит в строку ')]}', сравните с последним ожиданием, предварительно проверив непустой стек. Все остальные символы игнорируются по договорённости. При несовпадении сразу верните False.

### 4. Почему важен конец строки

После обхода незакрытые открытия остаются в стеке. Только пустой стек означает True. Время O(n): каждый символ рассматривается один раз. Память в худшем случае O(n), например когда вся строка состоит из открывающих скобок. Пустая строка сразу даёт True.

### Похожий случай, разобранный до самостоятельной задачи

1. Для 'x[{}]' буква x ничего не меняет; '[' добавляет ожидание ']', '{' добавляет '}'.
2. Символ '}' совпадает с последним ожиданием, затем ']' совпадает со следующим. Стек пуст.
3. Для '[}' ожидание ']' не совпадает с '}', поэтому дальнейшее чтение не исправит эту ошибку вложенности.

```python
stack = []
stack.append(']')
stack.append('}')
print(stack.pop())  # }
print(stack.pop())  # ]
print(not stack)    # True
```

### Самостоятельная задача

Напишите balanced(text). Проверьте правильную вложенность (), [], {} с помощью stack. Другие символы игнорируйте. Пустая строка правильная. Например, '{a:[()]}' → True, '([)]' → False. Обоснуйте O(n) времени и O(n) памяти в худшем случае.

In [ ]:
def balanced(text):
    stack = []
    opening = {"(": ")", "[": "]", "{": "}"}
    for char in text:
        # TODO: открывающий символ кладёт ожидаемую пару в stack;
        # закрывающий должен совпасть с последней ожидаемой парой.
        pass
    return len(stack) == 0
print(balanced("{a:[()]}"), balanced("([)]"))


In [ ]:
result = validate('brackets', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Возникает IndexError на первом закрывающем символе** — Проверьте, что стек не пуст, прежде чем вызывать pop.

**Строка '((' признана правильной** — После цикла нужно убедиться, что ни одного незакрытого ожидания не осталось.

**Объяснение для собеседования:** Какой инвариант стека отличает правильную вложенность от простого совпадения количеств?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def balanced(text):
    stack = []
    opening = {"(": ")", "[": "]", "{": "}"}
    for char in text:
        if char in opening:
            stack.append(opening[char])
        elif char in ")]}":
            if not stack or stack.pop() != char:
                return False
    return not stack
print(balanced("{a:[()]}"), balanced("([)]"))

```

</details>

## 16. Последний маршрут — финальные 15 минут

Поддерживать непрерывное окно без повторов и доказать, что левая граница не движется назад.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

### 1. Подстрока означает непрерывный фрагмент

Разрешено выбрать только подряд идущие символы исходной строки. Нельзя удалить повтор посередине и склеить оставшиеся части. Окно задаётся индексами left и right включительно; его длина right−left+1. Пустая строка имеет ответ 0.

### 2. Память о последнем появлении

Словарь last_seen хранит символ → последний индекс. enumerate(text) даёт индекс right и символ char. При появлении char второй раз нужно убрать из окна его предыдущую копию: передвинуть left как минимум на last_seen[char]+1. Затем обновляется последняя позиция char и лучшая длина.

### 3. Повтор может уже быть за пределами окна

Если предыдущий char находится левее текущего left, он уже не мешает. Поэтому новая граница равна max(left,last_seen[char]+1). Простое присваивание могло бы вернуть left назад и снова включить дубликаты. Индексы символов Unicode в Python относятся к символам строки, а не байтам UTF-8.

### 4. Почему алгоритм линейный

Правая граница посещает каждый символ один раз, левая никогда не возвращается назад. Поиск и запись dict обычно выполняются за постоянное ожидаемое время. Поэтому ожидаемая работа O(n), память зависит от числа разных символов, не превышая O(n). Корректность перехода важнее запоминания названия sliding window.

### Похожий случай, разобранный до самостоятельной задачи

1. Строка 'abcaef'. До второго a окно 'abc' имеет длину 3.
2. При a на позиции 3 прошлое a было на 0: left становится 1, окно 'bca'. Затем добавляются e и f: 'bcaef' имеет длину 5.
3. В случае 'abba' после второго b left=2. Последнее a встречалось на 0, уже вне окна: max сохраняет left=2.

```python
left = 2
last_seen = {'a':0, 'b':2}
char, right = 'a', 3
left = max(left, last_seen[char]+1)
length = right-left+1
print(left, length)  # 2, 2
```

### Самостоятельная задача

Напишите longest_unique(text): длина самого длинного непрерывного фрагмента без повторяющихся символов. Строка Unicode; пустая даёт 0. Примеры: 'abcabcbb'→3, 'bbbbb'→1. Начните с понятного корректного решения; затем используйте sliding window и last seen для O(n).

In [ ]:
def longest_unique(text):
    # TODO: окно [left, right] с уникальными символами.
    # Повтор сдвигает left только вправо.
    return 0
print(longest_unique("abcabcbb"))


In [ ]:
result = validate('longest_unique', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Для abba ответ больше 2** — Вероятно, left сдвинулась назад к предыдущей a. Используйте максимум старой границы и позиции после повтора.

**Считаются все уникальные символы строки** — Множество всей строки теряет ограничение непрерывности. Здесь нужен лучший непрерывный участок.

**Объяснение для собеседования:** Как объяснить переход left=max(left,last_seen+1) через инвариант окна?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def longest_unique(text):
    last_seen = {}
    left = best = 0
    for right, char in enumerate(text):
        if char in last_seen:
            left = max(left, last_seen[char] + 1)
        last_seen[char] = right
        best = max(best, right - left + 1)
    return best
print(longest_unique("abcabcbb"))

```

</details>

## 17. Один караван — одна выборка

Разделить связанные наблюдения по группам, сохранив независимость оценки и порядок записей.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

**Множество хранит уникальные значения**

set удаляет повторения и не хранит число повторов. set(['a','a','b']) содержит только a и b. x in values проверяет принадлежность, values.add(x) добавляет элемент. A & B даёт общие элементы. Это полезно для уникальных слов и групп; для частот нужен словарь, а не set.

```text
left = set(['a', 'a', 'b'])
right = {'b', 'c'}
common = left & right  # {'b'}, len(common) == 1
```

**Train, validation и test выполняют разные роли**

Train используют для изменения параметров модели. Validation измеряет поведение на других примерах и помогает выбрать настройки. Test оставляют для итоговой оценки после выбора. Обучение на validation labels переносит сведения об оценке в модель: сравнение перестаёт показывать качество на неизвестных ответах.

```text
Измерения растений A–F используются для обучения, G–H — для выбора настройки, I–J — для итоговой проверки. Повторные измерения одного растения держат в одной части.
```

### 1. Единица разбиения

Несколько строк могут относиться к одному растению, человеку или эксперименту. Если близкие записи одной группы попадут одновременно в train и validation, модель увидит сходные сведения заранее. В этой задаче поле group определяет единицу, которая целиком принадлежит одной части. Поле id просто отличает отдельные записи.

### 2. Сначала проверить план

validation_groups и test_groups задают принадлежность групп. Преобразуйте их в set, чтобы проверять in и пересечение. Если их пересечение непусто, одна группа назначена в две части — это ValueError до распределения. Не выбирайте молча одну из конфликтующих частей.

### 3. Распределение без изменения входа

Создайте новый словарь с тремя разными списками: train, validation, test. Пройдите rows слева направо. Если row['group'] в test_groups, добавьте запись в test; иначе если в validation_groups — в validation; остальные в train. list.append(row) сохраняет последовательность обхода. Сами поля row изменять не требуется.

### 4. Пустые группы и отсутствующие записи

Если rows пуст, все три результата остаются пустыми списками. Если заданная группа отсутствует в rows, добавлять искусственные записи нельзя. Допустимы строковые и целочисленные значения group: проверяйте принадлежность значения как есть, не меняя его тип.

### Похожий случай, разобранный до самостоятельной задачи

1. Записи идут в порядке A1(group=A), B1(group=B), A2(group=A), C1(group=C).
2. При validation_groups={B}, test_groups={C} train получает A1,A2, validation — B1, test — C1.
3. Обе записи A сохранились вместе и в исходном относительном порядке. Если добавить B ещё и в test_groups, план отвергается.

```python
validation_groups, test_groups = {'B'}, {'C'}
if validation_groups & test_groups:
    raise ValueError('Конфликт групп')
parts = {'train':[], 'validation':[], 'test':[]}
row = {'id':'B1', 'group':'B'}
parts['validation'].append(row)
print(parts)
```

### Самостоятельная задача

Напишите split_groups(rows,validation_groups,test_groups). Каждый row — dict с полями id и group. Все записи группы должны попасть в одну часть. Верните {'train':список,'validation':список,'test':список}, сохраняя исходный порядок. Пересечение validation_groups и test_groups — ValueError. Не меняйте rows.

In [ ]:
def split_groups(rows, validation_groups, test_groups):
    result = {"train": [], "validation": [], "test": []}
    # TODO: проверьте пересечение групп и распределите записи.
    return result
rows = [{"id": "a", "group": "north"}, {"id": "b", "group": "south"}]
print(split_groups(rows, {"south"}, set()))


In [ ]:
result = validate('split', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Перемешаны записи внутри результата** — Условие требует сохранить порядок. Проходите исходный список без сортировки.

**Одна группа попадает в разные части** — Решение принимается по полю group, а не независимо по id каждой строки.

**Объяснение для собеседования:** Почему случайное разбиение строк не всегда обеспечивает независимую validation выборку?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def split_groups(rows, validation_groups, test_groups):
    validation_groups, test_groups = set(validation_groups), set(test_groups)
    if validation_groups & test_groups:
        raise ValueError("Группа попала одновременно в validation и test")
    result = {"train": [], "validation": [], "test": []}
    for row in rows:
        key = "test" if row["group"] in test_groups else "validation" if row["group"] in validation_groups else "train"
        result[key].append(row)
    return result
rows = [{"id": "a", "group": "north"}, {"id": "b", "group": "south"}]
print(split_groups(rows, {"south"}, set()))

```

</details>

## 18. Индекс улик

Построить воспроизводимый лексический поиск и отличить релевантность от достоверности.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Словарь связывает ключ со значением**

dict хранит пары ключ → значение. В {'name':'test','count':2} ключи — строки name и count, а значения имеют разные типы. d['count'] читает значение, d['count']=3 заменяет его. Проверка key in d ищет ключ. d.get(key, default) возвращает запасное значение, если ключа нет. Один ключ хранит одно текущее значение; повторное присваивание его заменяет.

```text
positions = {}
positions['a'] = 0
positions['a'] = 3
# positions['a'] == 3; ('b' in positions) is False
```

**Множество хранит уникальные значения**

set удаляет повторения и не хранит число повторов. set(['a','a','b']) содержит только a и b. x in values проверяет принадлежность, values.add(x) добавляет элемент. A & B даёт общие элементы. Это полезно для уникальных слов и групп; для частот нужен словарь, а не set.

```text
left = set(['a', 'a', 'b'])
right = {'b', 'c'}
common = left & right  # {'b'}, len(common) == 1
```

### 1. Контракт учебного индекса

Каждый document содержит id и text. Запрос и тексты переводятся в нижний регистр через .lower(), затем .split() без аргумента разбивает строку по пробельным разделителям. Пунктуация здесь не очищается, синонимы не объединяются. set удаляет повторения слов: частота не влияет на score.

### 2. Как считается score

Пересечение query_tokens & document_tokens оставляет только общие слова. len этого множества — score. Документы с нулевым score отбрасываются. Это простейший lexical baseline: он оценивает совпадение токенов, не смысл ответа и не истинность текста.

### 3. Порядок при равенстве

Нужны убывающий score и возрастающий id при равенстве. Python сравнивает кортежи слева направо. Пара (-score,doc_id) позволяет обычной sorted отсортировать больший score раньше, а одинаковые score — по id. Срез [:k] выбирает первые k элементов. Если k<=0, условие этого упражнения возвращает пустой результат.

### 4. Вернуть только идентификаторы

Внутренние пары нужны для сортировки, но результат retrieve — список ID. Можно пройти отсортированные пары циклом и добавить второй элемент. Переменная _ часто обозначает компонент пары, который не нужен. Retrieval выбирает документы; формирование ответа моделью — следующий отдельный шаг RAG.

### 5. Ограничение найденного источника

Совпадение слов может привести к устаревшему, нерелевантному или недостоверному документу. В этом упражнении оценивается корректность индекса по заданному контракту. Для реальной системы дополнительно нужны проверка источников, политика обновления, качество retrieval и оценка конечных ответов.

### Похожий случай, разобранный до самостоятельной задачи

1. Запрос 'светлый мост'; документы b:'светлый мост мост', a:'мост у реки', c:'тёмный мост'.
2. У b score=2, у a и c score=1. Повтор слова 'мост' у b не добавляет балл.
3. При k=2 порядок ID — b,a. При пустом запросе общих слов нет и результат пуст.

```python
pairs = [(-1,'c'), (-2,'b'), (-1,'a')]
ordered = sorted(pairs)
answer = [document_id for score,document_id in ordered[:2]]
print(answer)  # ['b','a']
```

### Самостоятельная задача

Напишите retrieve(query,documents,k). document содержит id и text. Для простого lexical retrieval: lowercase, split по пробелам, уникальные токены; score — количество общих токенов с query. Оставьте score>0, отсортируйте по убыванию score, при равенстве по id и верните первые k ID. Это учебный baseline, не полноценный semantic search.

In [ ]:
def retrieve(query, documents, k):
    query_tokens = set(query.lower().split())
    # TODO: пересечение множеств, сортировка, top-k.
    return []
docs = [{"id": "a", "text": "синий фонарь"}, {"id": "b", "text": "красный фонарь у ворот"}]
print(retrieve("синий фонарь", docs, 1))


In [ ]:
result = validate('retrieval', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Повтор слова повышает score** — В контракте используются уникальные токены. Превратите списки слов в множества.

**При равных score результат зависит от входного порядка** — В условии задан tie-break по id. Добавьте id в ключ сравнения.

**Объяснение для собеседования:** Какие ошибки RAG могут остаться даже при безошибочно реализованном lexical retrieval?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def retrieve(query, documents, k):
    query_tokens = set(query.lower().split())
    scored = []
    for doc in documents:
        score = len(query_tokens & set(doc["text"].lower().split()))
        if score > 0:
            scored.append((-score, doc["id"]))
    return [doc_id for _, doc_id in sorted(scored)[:max(0,k)]]
docs = [{"id": "a", "text": "синий фонарь"}, {"id": "b", "text": "красный фонарь у ворот"}]
print(retrieve("синий фонарь", docs, 1))

```

</details>

## 19. Один луч внимания

Реализовать один вычисляемый шаг внимания и проследить соответствие форм Q, K и V.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Tensor, форма и тип**

torch.Tensor — массив чисел, над которым PyTorch выполняет операции. shape перечисляет длины осей: [3,2] — три строки по два числа. В этой игре первая ось обычно batch — несколько примеров, последняя features — признаки одного примера. float32 хранит дробные числа; int64 — целые, например индексы классов. device='cpu' оставляет вычисления на процессоре.

```text
import torch
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape)  # torch.Size([2, 2])
```

**Граф вычислений и градиент**

Если tensor создан с requires_grad=True, PyTorch записывает, какие операции привели к результату. Производная показывает локальную скорость изменения результата при изменении одного входа. Градиент собирает такие производные для нескольких чисел. Для одного итогового числа y вызов y.backward() проходит по графу назад и накапливает производные в .grad исходных leaf tensors.

```text
x = torch.tensor(3., requires_grad=True)
y = x*x
y.backward()
print(x.grad)  # tensor(6.)
```

### 1. Что хранит каждый вход

q — один вектор запроса длины d. keys — n векторов описания позиций, каждый длины d. values — n векторов содержимого длины v. Поэтому их формы [d], [n,d], [n,v]. Число n у keys и values одинаково: строка ключа соответствует строке содержимого.

### 2. От скалярного произведения к scores

keys @ q умножает матрицу на вектор и возвращает n scores. Каждый score — сумма попарных произведений одного ключа и запроса. Деление на math.sqrt(d) меняет масштаб scores; d можно получить из q.shape[0]. Это scaled dot-product: операции идут в указанном порядке.

### 3. Softmax создаёт веса всех позиций

torch.softmax(scores,dim=0) превращает n scores в неотрицательные веса с суммой 1. Больший score даёт больший вес, но остальные обычно не обнуляются. Стабильная реализация PyTorch сама обрабатывает экспоненты; вручную создавать Python-список или считать argmax не требуется.

### 4. Взвешенная сумма содержимого

weights @ values умножает [n] на [n,v] и возвращает [v]. Это сумма всех строк values, каждая умножена на свой вес. Argmax выбрал бы только одну строку и потерял смешивание. Используйте torch-операции напрямую, чтобы результат сохранял граф к q и параметрам будущей сети.

### 5. Граница маленького опыта

Здесь одна query и одна attention head, без маски. В self-attention Q,K,V получаются проекциями одной последовательности; multi-head использует несколько таких проекций и затем объединяет результаты. Реализация одной head помогает понять вычисление, но ещё не является полноценным Transformer.

### Похожий случай, разобранный до самостоятельной задачи

1. При q=[1,0], keys=[[1,0],[0,1]] raw scores равны [1,0], затем делятся на sqrt(2).
2. Softmax даёт примерно [0.67,0.33]. Если values=[[2,0],[0,4]], результат примерно [1.34,1.32].
3. В ответ входят обе строки values. Самая похожая позиция получила больше веса, но не вытеснила вторую полностью.

```python
import torch, math
query = torch.tensor([1.,0.], requires_grad=True)
keys = torch.tensor([[1.,0.],[0.,1.]])
contents = torch.tensor([[2.,0.],[0.,4.]])
scores = (keys @ query)/math.sqrt(query.shape[0])
weights = torch.softmax(scores, dim=0)
print(weights, weights @ contents)
```

### Самостоятельная задача

Напишите attend(q,keys,values): q имеет shape [d], keys [n,d], values [n,v]. Верните weighted sum values с весами softmax((keys @ q)/sqrt(d)), shape [v]. Не делайте argmax: attention смешивает values. Нужны настоящие torch операции с gradient.

In [ ]:
import torch
import math
def attend(q, keys, values):
    # TODO: scores → scaled scores → softmax по токенам → weighted sum.
    return values[0]
q = torch.tensor([1., 0.])
keys = torch.tensor([[1., 0.], [0., 1.]])
values = torch.tensor([[10., 0.], [0., 10.]])
print(attend(q, keys, values))


In [ ]:
result = validate('attention', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Возвращена строка values с максимальным score** — Attention в этой задаче — взвешенная сумма, не дискретный поиск argmax.

**Результат потерял gradient** — Создание нового tensor из вычисленных Python-чисел может разорвать граф. Сохраняйте вычисление в torch.

**Объяснение для собеседования:** Почему Q/K определяют веса, V определяет смешиваемое содержимое, а итог не равен обязательному выбору одного токена?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
import torch
import math
def attend(q, keys, values):
    scores = (keys @ q) / math.sqrt(q.shape[-1])
    weights = torch.softmax(scores, dim=0)
    return weights @ values
q = torch.tensor([1., 0.])
keys = torch.tensor([[1., 0.], [0., 1.]])
values = torch.tensor([[10., 0.], [0., 10.]])
print(attend(q, keys, values))

```

</details>

## 20. Склон ошибки

Вычислить один шаг градиентного спуска и объяснить его знак и масштаб.

### Что понадобится

**Функция, аргумент и результат**

Функция — именованный способ выполнить действия над переданными данными. В def double(x): имя x обозначает аргумент внутри функции. Строки тела имеют отступ. return передаёт результат вызывающему коду и завершает этот вызов; print только показывает текст. Поэтому проверяющей программе нужен return, даже если print уже вывел правильное число.

```text
def double(x):
    return 2*x
answer = double(4)  # answer == 8
```

**Последовательность и накопление**

Список хранит элементы в порядке. for value in values последовательно связывает value с каждым элементом. Переменная, созданная перед циклом, сохраняет накопленный результат между итерациями. if выполняет ветку при истинном условии; elif проверяет следующий случай. == сравнивает значения, а = присваивает. and требует выполнения обоих условий; or — хотя бы одного.

```text
total = 0
for value in [2, 5, 1]:
    total += value  # то же, что total = total + value
# total == 8
```

**Средняя квадратичная ошибка**

Для каждой пары прогноз–ответ берут разность, возводят её в квадрат и усредняют по всем элементам. Квадрат делает знак ошибки неважным и сильнее выделяет большие отклонения. MSE = sum((prediction-target)**2)/число элементов. Это число качества прогноза; оно не является моделью или алгоритмом обучения.

```text
Прогнозы [2,5], ответы [3,3]: ошибки [-1,2], квадраты [1,4], MSE = (1+4)/2 = 2.5.
```

### 1. Одна модель и одна цель

Здесь y_hat=w*x без свободного члена. Loss — среднее, а не сумма квадратов ошибок. Коэффициент w — изменяемый параметр. xs,ys — данные, которые остаются неизменными. lr — learning rate, множитель величины шага.

### 2. Откуда берётся формула производной

Для одного примера квадрат ошибки равен (w*x−y)². Производная квадрата даёт 2*(w*x−y), а изменение внутреннего выражения по w добавляет множитель x. Для MEAN-loss усредняются все такие вклады: gradient=2*sum((w*x−y)*x)/len(xs). Смысл градиента — локальное направление роста loss.

### 3. Почему градиент вычитается

Чтобы двигаться в сторону уменьшения loss, выполняют new_w=w−lr*gradient. Положительный gradient уменьшает w, отрицательный увеличивает. Слишком большой lr может перескочить область хороших значений; правильная формула не гарантирует улучшения при любом размере шага.

### 4. Реализация и граничные случаи

zip(xs,ys) даёт пары, sum складывает вклады, len(xs) — число примеров. Если xs пуст, условие требует вернуть исходный w без деления. При lr=0 параметры также не меняются. Функция возвращает новое число; она не меняет xs или ys и не подставляет известный правильный вес вручную.

### Похожий случай, разобранный до самостоятельной задачи

1. Пусть w=1, xs=[1,2], ys=[3,6]. Прогнозы [1,2], ошибки [−2,−4].
2. Произведения ошибки на x равны [−2,−8], gradient=2*(−10)/2=−10.
3. При lr=0.1 новый вес 2. Он сдвинулся в сторону уменьшения ошибки, но ещё не равен точному весу 3.

```python
weight, gradient, learning_rate = 1., -10., .1
updated = weight-learning_rate*gradient
print(updated)  # 2.0
```

### Самостоятельная задача

Напишите descend(w,xs,ys,lr): один шаг gradient descent для MEAN((w*x-y)^2), модель без bias. Производная = 2*MEAN((w*x-y)*x). Верните новое w = w-lr*gradient. Пустой набор оставляет w без изменений. Сначала предскажите направление движения, затем запустите.

In [ ]:
def descend(w, xs, ys, lr):
    # TODO: вычислите gradient и вычтите lr*gradient.
    return w
w = 0.0
for _ in range(5):
    w = descend(w, [1., 2.], [2., 4.], 0.1)
    print(w)


In [ ]:
result = validate('gradient_descent', globals(), variant='practice')
for case in result['results']:
    print(('✓ ' if case['passed'] else '✗ ') + case['name'] + ': ' + case['detail'])
print(result['feedback'])
if result.get('artifact'):
    for observation in result['artifact'].get('observations', []):
        print(observation)
# После самостоятельного объяснения замените practice на transfer для других входов.


### Проверка причин и ошибок

**Градиент прибавлен к весу** — Градиент указывает рост loss. Для спуска вычитайте lr*gradient.

**Нет деления на число примеров** — В этом упражнении MEAN-loss. Для SUM-loss масштаб производной был бы другим.

**Объяснение для собеседования:** Как различить ошибку знака градиента и слишком большой learning rate по изменению loss?

<details><summary>Эталон — после самостоятельной попытки</summary>

```python
def descend(w, xs, ys, lr):
    if not xs:
        return w
    gradient = 2 * sum((w*x-y)*x for x, y in zip(xs, ys)) / len(xs)
    return w - lr*gradient
w = 0.0
for _ in range(5):
    w = descend(w, [1., 2.], [2., 4.], 0.1)
    print(w)

```

</details>